In [ ]:
!pip install -q lifelines scikit-survival pyarrow  # wheels available for Colab

In [ ]:
!pip install xgboost==3.0.3 #..this version is the one which works with SHAP.

In [ ]:
import xgboost as xgb
xgb.__version__

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.preprocessing import QuantileTransformer, PowerTransformer, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from lifelines import CoxPHFitter
from sklearn.model_selection import train_test_split
from sksurv.util import Surv
from sksurv.ensemble import ComponentwiseGradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
#define the development path.
dev_path = '/path/to/your/project/'

#Load the Data

In [ ]:
#load the training, validation and testing cohorts.
train_data = pd.read_csv(os.path.join(dev_path, 'Cohorts time varying/Models 3 and 4/Modeling Cohorts/train_data_model3_imputed_capped.csv'), index_col=0)
val_data = pd.read_csv(os.path.join(dev_path, 'Cohorts time varying/Models 3 and 4/Modeling Cohorts/valid_data_model3_imputed_capped.csv'), index_col=0)
test_data = pd.read_csv(os.path.join(dev_path, 'Cohorts time varying/Models 3 and 4/Modeling Cohorts/test_data_model3_imputed_capped.csv'), index_col=0)

In [ ]:
def create_TOE_variable(df, outcome_age_col='Dementia'):
  '''
  This function creates a time-to-event variable using AF_age and 'Age at Last Follow-up' for controls
  and 'Dementia_age' for Dementia patients.
  '''
  # start counting at AF diagnosis age.
  t0 = df["AF_Age"] #..this is the starting time for each patient. Every patient will have a AF_age.

  df["time"]  = np.where(df[outcome_age_col].notna(),
                        df[outcome_age_col] - t0,
                        df["Age at Last Follow-up"] - t0)

  #df["event"] = df[outcome_age_col].notna().astype(int)
  df["event"] = df["Outcome"]                              #...outcome has already been created.

  return df

def count_drop_neg_time_to_event(df):
  '''
  This function counts the number of rows with negative time-to-event and drops them.
  '''
  df = df.copy() #..copy the dataframe to not corrupt the original one.

  # Step 1: Count rows with time < 0
  num_negative_time = (df['time'] < 0).sum()
  print(f"Number of rows with time < 0: {num_negative_time}")

  # Step 2: Drop rows with time < 0
  df = df[df['time'] >= 0].copy()

  # Step 3: Recount number of events
  event_counts = df['event'].value_counts().sort_index()

  print("\nEvent distribution after removing invalid times:")
  for event_value, count in event_counts.items():
      print(f"Event = {event_value}: {count} rows")

  return df

def create_X_y(df, vars_list):
  '''
  Create the X and y variables for time-varying risk prediction modeling.
  ***Select the variables list by physically inspecting the full list of variables.
  '''

  X   = df[vars_list].astype(float)
  y   = df[["event", "time"]]

  return X, y


In [ ]:
#...create the time and event variables required for time-to-event analysis.
train_data = create_TOE_variable(train_data, outcome_age_col='NeuroDx_Earliest_Age').drop(columns='Outcome')
val_data = create_TOE_variable(val_data, outcome_age_col='NeuroDx_Earliest_Age').drop(columns='Outcome')
test_data = create_TOE_variable(test_data, outcome_age_col='NeuroDx_Earliest_Age').drop(columns='Outcome')

####it seems that there are some instances where time is negative. remove these entries.

In [ ]:
#...count the number of negative time-to-event and drop them.
#...then recount the number of dementia and controls in the different datasets.
print("Training Data:")
train_data_new = count_drop_neg_time_to_event(train_data)
print("\n")
print("Validation Data:")
val_data_new = count_drop_neg_time_to_event(val_data)
print("\n")
print("Test Data:")
test_data_new = count_drop_neg_time_to_event(test_data)

In [ ]:
#save these datasets
train_data_new.to_csv(os.path.join(dev_path, 'Cohorts time varying/Models 3 and 4/Modeling Cohorts/train_data_imputed_TE_model4.csv'))
val_data_new.to_csv(os.path.join(dev_path, 'Cohorts time varying/Models 3 and 4/Modeling Cohorts/valid_data_imputed_TE_model4.csv'))
test_data_new.to_csv(os.path.join(dev_path, 'Cohorts time varying/Models 3 and 4/Modeling Cohorts/test_data_imputed_TE_model4.csv'))

#Reload the Time-to-Event data created

In [ ]:
train_data_new = pd.read_csv(os.path.join(dev_path, 'Cohorts time varying/Models 3 and 4/Modeling Cohorts/train_data_imputed_TE_model3.csv'), index_col=0)
val_data_new = pd.read_csv(os.path.join(dev_path, 'Cohorts time varying/Models 3 and 4/Modeling Cohorts/valid_data_imputed_TE_model3.csv'), index_col=0)
test_data_new = pd.read_csv(os.path.join(dev_path, 'Cohorts time varying/Models 3 and 4/Modeling Cohorts/test_data_imputed_TE_model3.csv'), index_col=0)

In [ ]:
list(train_data_new.columns)

In [ ]:
#define the final list of predictors.
#basic_remove_vars = ['PID', 'Dementia', 'Age at Last Follow-up', 'AF_Age', 'AF_age_1', 'Dementia_age', 'time', 'event'] #...these should be removed because they are related with the outcome.
basic_remove_vars = ['PID', 'Age at Last Follow-up', 'NeuroDx_Earliest_Age', 'time', 'event']

#...the following variables were found non-significant in a univariate feature selection (p > 0.1)
#...although some lab variables were insignificant they were retained because the 'missing' flag variable would be meaningless otherwise.
'''
stat_remove_vars = ['CAD', 'CKD', 'Cancer', 'HIV_AIDS', 'Alcohol_Abuse',
                    'B12_Deficiency', 'Lupus', 'Aspirin', 'P2Y12',
                    'CCB', 'Digoxin', 'Sotalol', 'Dofetilide', 'Anticholinergics',
                    'Antipsychotics', 'Opioids', 'Glucocorticoids', 'Congenital_Procedure',
                    'Pacemaker_Procedure', 'ICD_Procedure', 'CRT',
                    'Chloride Value 3mo', 'Creatinine Value 3mo', 'HCT Value 3mo', 'Taxis Value 3mo']
'''
stat_remove_vars = {
    1: ['Alcohol_Abuse', 'Chloride Value 3mo', 'HCT Value 3mo', 'HIV_AIDS', 'P2Y12',
        'MI', 'Taxis Value 3mo', 'Anticholinergics', 'Lupus', 'Digoxin', 'B12_Deficiency',
        'CAD', 'Dofetilide', 'Aspirin', 'Pacemaker_Procedure', 'Congenital_Procedure',
        'CCB', 'Opioids', 'Cancer', 'LMW_Heparin_Analogs', 'CRT', 'DM', 'ADI_NATRANK', 'ADI_STATERANK'],

    2: ['ADI_NATRANK', 'ADI_STATERANK', 'CAD', 'MI', 'VT', 'HIV_AIDS', 'Schizophrenia',
        'Aspirin', 'P2Y12', 'LMW_Heparin_Analogs', 'CCB', 'Digoxin', 'Sotalol', 'Dofetilide',
        'GLP1_Agonists', 'Glucocorticoids', 'Congenital_Procedure', 'Pacemaker_Procedure',
        'Chloride Value 3mo', 'Taxis Value 3mo', 'ADI_NATRANK_missing', 'ADI_STATERANK_missing'],

    3: ['HIV_AIDS', 'Alcohol_Abuse', 'P2Y12', 'MI', 'Taxis Value 3mo', 'HCT Value 3mo', 'Anticholinergics',
        'Chloride Value 3mo', 'Pacemaker_Procedure', 'Dofetilide', 'CAD', 'CCB', 'Opioids', 'Lupus',
        'B12_Deficiency', 'Congenital_Procedure', 'Digoxin', 'Aspirin', 'ALS_FTD_Spectrum', 'Epilepsy',
        'Cancer', 'LMW_Heparin_Analogs', 'CRT', 'ICD_Procedure', 'Sotalol', 'ADI_NATRANK', 'Dronedarone', 'ADI_STATERANK'],

    4: ['ethnic', 'Taxis Value 3mo', 'Sedatives', 'HIV_AIDS', 'Dofetilide', 'Chloride Value 3mo', 'Opioids',
        'Alcohol_Abuse', 'CCB', 'B12_Deficiency', 'Digoxin', 'MI', 'Congenital_Procedure', 'ADI_NATRANK',
        'Pacemaker_Procedure', 'Antipsychotics', 'COPD', 'ALS_FTD_Spectrum', 'Epilepsy', 'ADI_STATERANK',
        'Cancer', 'INR_missing', 'Sotalol', 'Lupus', 'Coumadin_warfarin', 'CKD', 'Anticholinergics', 'CAD',
        'LMW_Heparin_Analogs', 'Calcium Value 3mo']
}

model = 3                                                          #...change the model number as the static variables to be removed will change.
total_remove_vars = basic_remove_vars + stat_remove_vars[model]    #...all variables to be removed.

vars_list = [s for s in list(train_data_new.columns) if s not in total_remove_vars] #...the remaining variables.

In [ ]:
def create_X_y(df, vars_list):
  '''
  Create the X and y variables for time-varying risk prediction modeling.
  ***Select the variables list by physically inspecting the full list of variables.
  '''

  X   = df[vars_list].astype(float)
  y   = df[["event", "time"]]

  return X, y

In [ ]:
#...create X and y for model development and hyper-parameter tuning.
X_train, y_train = create_X_y(train_data_new, vars_list)
X_val, y_val = create_X_y(val_data_new, vars_list)
X_test, y_test = create_X_y(test_data_new, vars_list)

#Model Development

#####XGBoost-Trees

In [ ]:
#=======This function is for saving and loading trained/tuned models=======

import json, xgboost as xgb
from typing import Optional, Dict, Tuple, List
import pandas as pd

def save_xgb_cox_model(
    model_final: xgb.Booster,
    prefix: str = "xgb_cox_gpu",
    *,
    selected_cols: Optional[List[str]] = None,
    X_train: Optional[pd.DataFrame] = None,
    best_params: Optional[Dict] = None,
    best_round: Optional[int] = None,
    extra_meta: Optional[Dict] = None
) -> None:
    """
    Save trained XGBoost Cox model and metadata (feature names, params, etc.).
    One of selected_cols or X_train must be provided to freeze column order.
    """
    # Determine the feature list we will lock in
    if selected_cols is None and X_train is not None:
        selected_cols = X_train.columns.tolist()
    if selected_cols is None:
        # Try to fall back to names embedded in the model (not guaranteed)
        try:
            selected_cols = list(model_final.feature_names) if model_final.feature_names else None
        except Exception:
            selected_cols = None
    if not selected_cols:
        raise ValueError("selected_cols not provided and could not be inferred. "
                         "Pass selected_cols=list(X_train.columns).")

    # 1) save booster
    model_path = f"{prefix}.json"
    model_final.save_model(model_path)

    # 2) save metadata
    meta = {
        "feature_names": selected_cols,
        "best_params": best_params,
        "best_round": int(best_round) if best_round is not None else None,
        "xgboost_version": xgb.__version__,
    }
    if extra_meta:
        meta.update(extra_meta)

    meta_path = f"{prefix}_meta.json"
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"Saved model -> {model_path}")
    print(f"Saved meta  -> {meta_path}")


def load_xgb_cox_model(prefix: str = "xgb_cox_gpu") -> Tuple[xgb.Booster, Dict]:
    """
    Load trained XGBoost Cox model + metadata dict.
    Returns (booster, meta). Use meta['feature_names'] to align inputs.
    """
    model_path = f"{prefix}.json"
    meta_path = f"{prefix}_meta.json"

    booster = xgb.Booster()
    booster.load_model(model_path)

    with open(meta_path) as f:
        meta = json.load(f)

    return booster, meta


def align_features(df: pd.DataFrame, feature_names: List[str]) -> pd.DataFrame:
    """
    Reorder/trim columns to match training. Errors if required columns are missing.
    """
    missing = [c for c in feature_names if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in input: {missing}")
    return df.loc[:, feature_names]


In [ ]:
# ===========================
# XGBoost Cox (GPU) — Wide Randomized Search with Timing
# ===========================
import time, itertools, numpy as np, pandas as pd, xgboost as xgb
from sksurv.metrics import concordance_index_censored

# ---- Helpers ----
def to_xgb_cox_label(y_df: pd.DataFrame) -> np.ndarray:
    """Encode right-censored observations as negative times for XGBoost Cox."""
    t = y_df["time"].astype(float).to_numpy().copy()
    e = y_df["event"].astype(bool).to_numpy()
    t[~e] *= -1.0
    return t

def cindex_from_risk(y_df, risk):
    return concordance_index_censored(
        y_df["event"].astype(bool).values,
        y_df["time"].astype(float).values,
        np.asarray(risk).ravel()
    )[0]

def xgb_cox_gpu_random_search(
    X_train, y_train, X_val, y_val,
    X_test=None, y_test=None,
    n_iter=200,                    # <-- expand this if your GPU is cruising
    num_boost_round=8000,          # big cap; ES will stop early
    early_stopping_rounds=300,     # patience
    seed=42,
    grid=None,
    gpu_id=None
):
    """
    Randomized search for GPU-accelerated XGBoost Cox (gbtree) with early stopping.
    Returns (results_df, best_params, best_round, booster_final, test_cindex).
    """

    # Wide, sensible grid for regular trees (tweak freely)
    if grid is None:
        grid = {
            "max_depth":        [3, 5, 7, 9, 11],             # deeper trees allowed
            "min_child_weight": [1, 2, 3, 5, 8],
            "gamma":            [0.0, 0.05, 0.1, 0.2, 0.4],    # split loss reduction
            "subsample":        [0.6, 0.75, 0.9, 1.0],
            "colsample_bytree": [0.5, 0.7, 0.85, 1.0],
            "eta":              [0.01, 0.03, 0.05, 0.08, 0.1], # learning rate
            "lambda":           [0.0, 1.0, 3.0, 5.0, 10.0],    # L2
            "alpha":            [0.0, 1e-3, 1e-2, 0.1],        # L1
            "max_bin":          [128, 256, 512],               # histogram bins (GPU)
        }

    # Build all combos and sample n_iter at random
    keys = list(grid.keys())
    all_combos = list(itertools.product(*[grid[k] for k in keys]))
    rng = np.random.default_rng(seed)
    n_iter = min(n_iter, len(all_combos))
    idxs = rng.choice(len(all_combos), size=n_iter, replace=False)

    # DMatrix objects (no scaling needed for trees)
    dtr = xgb.DMatrix(X_train, label=to_xgb_cox_label(y_train))
    dva = xgb.DMatrix(X_val,   label=to_xgb_cox_label(y_val))
    dte = None
    if X_test is not None and y_test is not None:
        dte = xgb.DMatrix(X_test, label=to_xgb_cox_label(y_test))

    results = []
    best_c, best_params, best_round, best_bst = -np.inf, None, None, None

    for i, ix in enumerate(idxs, start=1):
        vals = all_combos[ix]
        params_trial = dict(zip(keys, vals))

        # Core Cox + GPU params
        xgb_params = {
            "objective":        "survival:cox",
            "booster":          "gbtree",
            "tree_method":      "gpu_hist",
            "predictor":        "gpu_predictor",
            "eval_metric":      "cox-nloglik",
            "verbosity":        0,
            "seed":             seed,
            # hyperparams from the trial:
            "max_depth":        params_trial["max_depth"],
            "min_child_weight": params_trial["min_child_weight"],
            "gamma":            params_trial["gamma"],
            "subsample":        params_trial["subsample"],
            "colsample_bytree": params_trial["colsample_bytree"],
            "eta":              params_trial["eta"],
            "lambda":           params_trial["lambda"],
            "alpha":            params_trial["alpha"],
            "max_bin":          params_trial["max_bin"],
        }
        if gpu_id is not None:
            xgb_params["gpu_id"] = int(gpu_id)

        t0 = time.time()
        bst = xgb.train(
            params=xgb_params,
            dtrain=dtr,
            num_boost_round=num_boost_round,
            evals=[(dva, "val")],
            early_stopping_rounds=early_stopping_rounds,
            verbose_eval=False
        )
        secs = time.time() - t0

        # Validation C-index on raw margins (risk scores)
        risk_val = bst.predict(dva, output_margin=True)
        c_val = cindex_from_risk(y_val, risk_val)
        br = getattr(bst, "best_iteration", None)

        # Log row
        row = {**params_trial,
               "best_round": int(br) if br is not None else np.nan,
               "val_cindex": float(c_val),
               "fit_time_s": float(secs)}
        results.append(row)

        print(f"[{i}/{n_iter}] params={params_trial} | best_round={br} | C-val={c_val:.3f} | {secs:.1f}s")

        # Track best
        if c_val > best_c:
            best_c, best_params, best_round, best_bst = c_val, params_trial.copy(), (int(br) if br is not None else None), bst

    results_df = pd.DataFrame(results).sort_values("val_cindex", ascending=False).reset_index(drop=True)

    # Refit best on TRAIN+VAL using the chosen #rounds
    X_trv = pd.concat([X_train, X_val], axis=0)
    y_trv = pd.concat([y_train, y_val], axis=0)
    dtrv = xgb.DMatrix(X_trv, label=to_xgb_cox_label(y_trv))

    refit_params = {
        "objective":   "survival:cox",
        "booster":     "gbtree",
        "tree_method": "gpu_hist",
        "predictor":   "gpu_predictor",
        "eval_metric": "cox-nloglik",
        "verbosity":   0,
        "seed":        seed,
        **best_params
    }
    if gpu_id is not None:
        refit_params["gpu_id"] = int(gpu_id)

    n_rounds = (best_round + 1) if best_round is not None else num_boost_round
    bst_final = xgb.train(refit_params, dtrv, num_boost_round=n_rounds, verbose_eval=False)

    test_cindex = None
    if dte is not None:
        risk_test = bst_final.predict(dte, output_margin=True)
        test_cindex = concordance_index_censored(
            y_test["event"].astype(bool).values,
            y_test["time"].astype(float).values,
            np.asarray(risk_test).ravel()
        )[0]
        print(f"\nTEST C-index (XGBoost Cox, GPU regular trees) = {test_cindex:.3f}")

    return results_df, best_params, best_round, bst_final, test_cindex


In [ ]:
res_df, best_params, best_round, model_final, c_test = xgb_cox_gpu_random_search(
    X_train, y_train, X_val, y_val,
    X_test, y_test,
    n_iter=200,                    # try 200–400 if it stays fast
    num_boost_round=8000,
    early_stopping_rounds=300,
    seed=42,
    gpu_id=0                       # or None
)

# Inspect / save the search results
from IPython.display import display
display(res_df.head(20))
res_df.to_csv("xgb_cox_gpu_search_results.csv", index=False)
print("Best params:", best_params)
print("Best round:", best_round)
print("Test C-index:", c_test)


In [ ]:
save_path = "/content/xgb_cox_gpu_random_search_results_model2.csv"
res_df.sort_values("val_cindex", ascending=False) #..sort according to decreasing C-index values.
res_df.to_csv(save_path, index=False)
print("Saved to:", save_path)


In [ ]:
# Suppose these exist from your run
# model_final  -> xgboost.Booster
# best_params  -> dict
# best_round   -> int or None
# X_train      -> the exact DataFrame used to train (after filtering/selection)

save_xgb_cox_model(
    model_final,
    prefix="xgb_cox_gpu_model2",
    X_train=X_train,                 # this freezes column order
    best_params=best_params,
    best_round=best_round
)


In [ ]:
bst, meta = load_xgb_cox_model("xgb_cox_gpu_model3") #...change the model number here.

# Get the frozen feature list and align your new data
selected_cols = meta["feature_names"]
Xte_aligned = align_features(X_test, selected_cols)

# Predict risk scores (log-hazard; higher = higher risk)
dte = xgb.DMatrix(Xte_aligned)
risk_test = bst.predict(dte, output_margin=True)

print("Loaded XGBoost version:", meta.get("xgboost_version"))
print("Best round used:", meta.get("best_round"))
print("Optional thresholds:", meta.get("dca_thresholds"))

In [ ]:
import numpy as np, xgboost as xgb
from sksurv.metrics import concordance_index_censored

# Assumes you already have:
#   bst            -> loaded xgboost.Booster (your trained model)
#   X_test, y_test -> DataFrames (y_test has ["event","time"])
#   meta           -> dict with "feature_names" and (optionally) "test_cindex"
# If you don't have `meta`, set `feature_names = selected_cols` and `c_test_ref = ...` manually.

feature_names = meta.get("feature_names", None)  # or use your `selected_cols`
if feature_names is None:
    raise ValueError("No feature list found. Provide `selected_cols` or `meta['feature_names']`.")

# Align columns/order
missing = [c for c in feature_names if c not in X_test.columns]
if missing:
    raise ValueError(f"Missing columns in X_test: {missing}")
Xte = X_test.loc[:, feature_names]

# Predict risk (raw margin = log-hazard)
dte = xgb.DMatrix(Xte)
risk_test = bst.predict(dte, output_margin=True)

# Compute C-index on TEST
c_loaded = concordance_index_censored(
    y_test["event"].astype(bool).values,
    y_test["time"].astype(float).values,
    np.asarray(risk_test).ravel()
)[0]
print(f"[SANITY] Loaded model TEST C-index = {c_loaded:.6f}")

# Compare to reference C-index if available
c_test_ref = meta.get("test_cindex", None)  # or set c_test_ref = 0.xxx if you logged it earlier
if c_test_ref is not None:
    diff = abs(c_loaded - float(c_test_ref))
    print(f"[SANITY] Reference = {float(c_test_ref):.6f} | Δ = {diff:.6g} | {'MATCH' if diff <= 1e-9 else 'MISMATCH'}")


In [ ]:
# =========================================================
# 95% CI for TEST C-index via bootstrap resampling
# =========================================================
import numpy as np
from sksurv.metrics import concordance_index_censored

# Reuse:
#   y_test["event"], y_test["time"], risk_test
# from the previous cell

event = y_test["event"].astype(bool).to_numpy()
time  = y_test["time"].astype(float).to_numpy()
risk  = np.asarray(risk_test).ravel()

n_boot = 1000
rng = np.random.default_rng(42)

boot_c = []

n = len(y_test)
idx_all = np.arange(n)

for b in range(n_boot):
    # sample test patients with replacement
    idx = rng.choice(idx_all, size=n, replace=True)

    e_b = event[idx]
    t_b = time[idx]
    r_b = risk[idx]

    # skip degenerate resamples with no event variation
    # concordance_index_censored can fail or be meaningless here
    if e_b.sum() == 0:
        continue
    if np.all(t_b == t_b[0]):
        continue

    try:
        c_b = concordance_index_censored(e_b, t_b, r_b)[0]
        if np.isfinite(c_b):
            boot_c.append(c_b)
    except Exception:
        continue

boot_c = np.asarray(boot_c)

if len(boot_c) < 100:
    raise RuntimeError(
        f"Too few valid bootstrap samples ({len(boot_c)}). "
        "This can happen if the test set is very small or has very few events."
    )

ci_lower, ci_upper = np.percentile(boot_c, [2.5, 97.5])

print(f"TEST C-index = {c_loaded:.6f}")
print(f"95% bootstrap CI = ({ci_lower:.6f}, {ci_upper:.6f})")
print(f"Valid bootstrap samples = {len(boot_c)}/{n_boot}")

In [ ]:
# =========================================================
# Predict risk and compute C-index on VALIDATION set
# =========================================================

# Align validation columns/order
missing = [c for c in feature_names if c not in X_val.columns]
if missing:
    raise ValueError(f"Missing columns in X_val: {missing}")

Xva = X_val.loc[:, feature_names]

# Predict validation risk
dva = xgb.DMatrix(Xva)
risk_val = bst.predict(dva, output_margin=True)

# Compute validation C-index
c_val_loaded = concordance_index_censored(
    y_val["event"].astype(bool).values,
    y_val["time"].astype(float).values,
    np.asarray(risk_val).ravel()
)[0]

print(f"[SANITY] Loaded model VALIDATION C-index = {c_val_loaded:.6f}")

In [ ]:
# =========================================================
# 95% CI for VALIDATION C-index via bootstrap resampling
# =========================================================
import numpy as np
from sksurv.metrics import concordance_index_censored

# Reuse:
#   y_val["event"], y_val["time"], risk_val

event = y_val["event"].astype(bool).to_numpy()
time  = y_val["time"].astype(float).to_numpy()
risk  = np.asarray(risk_val).ravel()

n_boot = 1000
rng = np.random.default_rng(42)

boot_c = []

n = len(y_val)
idx_all = np.arange(n)

for b in range(n_boot):
    idx = rng.choice(idx_all, size=n, replace=True)

    e_b = event[idx]
    t_b = time[idx]
    r_b = risk[idx]

    if e_b.sum() == 0:
        continue
    if np.all(t_b == t_b[0]):
        continue

    try:
        c_b = concordance_index_censored(e_b, t_b, r_b)[0]
        if np.isfinite(c_b):
            boot_c.append(c_b)
    except Exception:
        continue

boot_c = np.asarray(boot_c)

if len(boot_c) < 100:
    raise RuntimeError(
        f"Too few valid bootstrap samples ({len(boot_c)}). "
        "This can happen if the validation set is very small or has very few events."
    )

ci_lower, ci_upper = np.percentile(boot_c, [2.5, 97.5])

c_val = concordance_index_censored(event, time, risk)[0]

print(f"VALIDATION C-index = {c_val:.6f}")
print(f"95% bootstrap CI = ({ci_lower:.6f}, {ci_upper:.6f})")
print(f"Valid bootstrap samples = {len(boot_c)}/{n_boot}")

######Plot feature importance

In [ ]:
import shap
#import xgboost as xgb
import matplotlib.pyplot as plt

In [ ]:
#...Run this if re-loading the trained models.
model_final = bst

In [ ]:
# --- SHAP global importance for XGBoost Cox (trees) ---
# Assumes:
#   model_final : xgboost.Booster (trained)
#   meta["feature_names"] or selected_cols : list of training column order
#   X_train, X_val : DataFrames

# 1) Build the background set from TRAIN+VAL (no test peeking)
feat_names = meta.get("feature_names", None)  # or: feat_names = selected_cols
if feat_names is None:
    raise ValueError("Need feature_names (meta['feature_names'] or selected_cols).")

X_bg = pd.concat([X_train, X_val], axis=0)
X_bg = X_bg.loc[:, feat_names].replace([np.inf, -np.inf], np.nan).dropna(axis=0)
bg = X_bg.sample(n=min(5000, len(X_bg)), random_state=1)

# 2) SHAP on raw margin (log-risk) for a Cox tree model
explainer = shap.TreeExplainer(model_final, feature_perturbation="interventional", model_output="raw")
shap_vals = explainer.shap_values(bg)   # shape: (n_samples, n_features)

# 3) Global importance = mean(|SHAP|) per feature
imp = pd.Series(np.abs(shap_vals).mean(axis=0), index=bg.columns).sort_values(ascending=False)
imp_df = imp.reset_index().rename(columns={"index":"feature", 0:"mean_abs_shap"})
imp_df.to_csv("shap_global_importance_trainval_model1.csv", index=False) #....CHANGE model number.

print("Top 20 features by mean|SHAP|:")
print(imp_df.head(20))

# 4) Nice plots (save at 300 dpi)
plt.figure(figsize=(8, max(4, 0.35*len(imp_df))))
imp_df.head(30).iloc[::-1].plot.barh(x="feature", y="mean_abs_shap", legend=False)
plt.xlabel("Mean |SHAP| (log-risk units)")
plt.ylabel("")
plt.title("Global importance (TRAIN+VAL)")
plt.tight_layout()
plt.savefig("shap_bar_trainval_model1.png", dpi=300, bbox_inches="tight") #...CHANGE model number.
plt.show()

# Optional: beeswarm for the same background sample
# (For large data, you can subsample bg again to ~2k rows)
shap.summary_plot(shap_vals, features=bg, show=False)
plt.title("SHAP beeswarm (TRAIN+VAL)")
plt.tight_layout()
plt.savefig("shap_beeswarm_trainval_model1.png", dpi=300, bbox_inches="tight") #...CHANGE model number
plt.show()


#Rank Features and find performance improvement by iteratively adding features

get the confidence intervals for all iterations

In [ ]:
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
imp = pd.Series(mean_abs_shap, index=feat_names).sort_values(ascending=False)
ranking = imp.index  # descending importance

# Means for ablation (neutral fill)
bg_means = X_bg.mean(axis=0)

# ---- 3) Evaluate C-index when keeping only top-k features
ks = [1, 3, 5, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40]
c_all = 0.823268  #...change this based on best performance of model in question
rows = [{"k": "all", "c_index": float(c_all), "delta_vs_all": 0.0}]

for k in ks:
    keep = set(ranking[:k])

    Xte_k = Xte.copy()
    # ablate (drop-in) all *non-keep* features to TRAIN+VAL means
    drop_cols = [c for c in feature_names if c not in keep]
    Xte_k.loc[:, drop_cols] = bg_means.loc[drop_cols].values

    d_k = xgb.DMatrix(Xte_k)
    risk_k = bst.predict(d_k, output_margin=True)
    c_k = concordance_index_censored(
        y_test["event"].astype(bool).values,
        y_test["time"].astype(float).values,
        np.asarray(risk_k).ravel()
    )[0]

    rows.append({"k": int(k), "c_index": float(c_k), "delta_vs_all": float(c_k - c_all)})

results = pd.DataFrame(rows)

# Pretty print
print("\n[Baseline] TEST C-index (all features): {:.6f}".format(c_all))
print("\n[Ranking] Top 20 by mean |SHAP| on TRAIN+VAL:")
print(imp.head(20))

print("\n[C-index after keeping only top-k (others mean-ablated)]")
print(results)

In [ ]:
# Keep only numeric k values and sort by k
df_plot = results[results["k"] != "all"].copy()
df_plot = df_plot.sort_values("k")

plt.figure(figsize=(6,4))
plt.plot(df_plot["k"].values, df_plot["c_index"].values, marker="o")
plt.xlabel("Most important features", fontsize=14)
plt.ylabel("C-index", fontsize=14)
plt.title("C-index vs top features", fontsize=14)
plt.savefig("Ablation plot top k features and C-index.png", dpi=300)
plt.show()

In [ ]:
df_plot.head()

#Find the Optimal number of features

#####AIC and BIC

In [ ]:
# BIC is required for picking the optimal number of features.
def cox_partial_bic(cph, df_fit, event_col):
    """
    Compute BIC for a fitted lifelines CoxPHFitter using the partial likelihood.
    Convention: use the number of events as 'n' for BIC.

    Parameters
    ----------
    cph : lifelines.CoxPHFitter (already fitted)
    df_fit : pandas.DataFrame used for the fit (or a view with event_col)
    event_col : str, name of the event indicator column (1=event, 0=censored)

    Returns
    -------
    float : BIC value (-2*loglik + p*ln(n_events))
    """
    loglik = cph.log_likelihood_               # partial log-likelihood at MLE
    p = cph.params_.shape[0]                   # number of coefficients
    n_events = int(df_fit[event_col].sum())    # number of observed events
    if n_events <= 0:
        raise ValueError("BIC needs at least one event.")
    return -2.0 * loglik + p * np.log(n_events)

In [ ]:
print("The total number of variables in the model is:", len(list(imp.index)))

In [ ]:
df = train_data_new.copy()

In [ ]:
# pick the optimal number of features.
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter

def select_k_by_ic(
    df,
    feature_ranking,            # list of features in descending importance
    duration_col,               # e.g., "time_to_event"
    event_col,                  # e.g., "event"
    k_grid=None,                # iterable of k values; if None, use 1..len(feature_ranking)
    penalizer=0.0,              # optional L2 penalizer to stabilize fits
    strata=None,                # optional list of columns for stratification
    show_warnings=False         # set True to inspect fit warnings
):
    """
    Returns a DataFrame with log-likelihood, AIC, BIC per k and the optimal ks.
    Uses CoxPH partial likelihood (standard for AIC/BIC with Cox models).
    """
    if k_grid is None:
        k_grid = range(1, len(feature_ranking) + 1)
    else:
        k_grid = list(k_grid)

    records = []
    for k in k_grid:
        cols = list(feature_ranking[:k]) + [duration_col, event_col]
        # drop rows with any NA in the selected columns
        d = df[cols]
        print("the total number of columns are:", len(d.columns))

        # lifelines expects float columns for covariates
        covars = feature_ranking[:k]
        #for c in covars:
        #    if not np.issubdtype(d[c].dtype, np.number):
        #        d[c] = pd.to_numeric(d[c], errors="coerce")
        #d = d.dropna(axis=0)

        #if len(d) == 0:
        #    continue

        cph = CoxPHFitter(penalizer=penalizer)

        cph.fit(
            d[covars + [duration_col, event_col]],
            duration_col=duration_col,
            event_col=event_col,
            strata=strata
        )
        # lifelines exposes partial log-likelihood and ICs
        loglik = cph.log_likelihood_
        aic = cph.AIC_partial_
        #bic = cph.BIC_partial_  #BIC_partial is not present in the cph from lifelines.
        bic = cox_partial_bic(cph, d, event_col)
        n_params = len(covars)  # number of coefficients
        n_events = int(d[event_col].sum())

        records.append({
            "k": k,
            "loglik_partial": loglik,
            "AIC": aic,
            "BIC": bic,
            "n_params": n_params,
            "n_events": n_events
        })

    out = pd.DataFrame.from_records(records).sort_values("k").reset_index(drop=True)
    if out.empty:
        raise RuntimeError("No models were successfully fit; check data and feature names.")

    # Optimal ks (lower is better)
    k_aic = out.loc[out["AIC"].idxmin(), "k"]
    k_bic = out.loc[out["BIC"].idxmin(), "k"]

    return out, k_aic, k_bic

# ----------------------------
# Example usage
# ----------------------------
# df: your dataframe with columns = features + duration_col + event_col
# feature_ranking: list of features sorted by importance (most→least)
# duration_col = "time"
# event_col    = "event"

# Suppose you want to evaluate exactly the k values shown in your plot:
k_grid = [5, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50]

#df=df.drop(columns=['PID', 'AF_Age', 'Age at Last Follow-up', 'Dementia', 'Dementia_age'])  #..OLD model.
df = df.drop(columns=['PID', 'Age at Last Follow-up', 'NeuroDx_Earliest_Age'])   #..NEW models

out_df, best_k_aic, best_k_bic = select_k_by_ic(
    df=df,
    feature_ranking=list(imp.index),
    duration_col="time",
    event_col="event",
    k_grid=k_grid, #...cover all the variables in the XGBoost model.
    penalizer=0.0
)
print("Best k by AIC:", best_k_aic)
print("Best k by BIC:", best_k_bic)
display(out_df)

In [ ]:
print("The optimal feature list selected by Bayesian Information Criteria (BIC) is:", list(imp.index)[:20])

In [ ]:
import matplotlib.pyplot as plt

# Keep only k ≤ 40 for both series, and sort by k
df_c = df_plot[df_plot["k"] <= 40].sort_values("k").copy()
df_b = out_df[out_df["k"] <= 40].sort_values("k").copy()

# Find k where BIC is minimum (within k ≤ 40)
k_bic_min = df_b.loc[df_b["BIC"].idxmin(), "k"]

fig, ax1 = plt.subplots(figsize=(7,4), constrained_layout=True)  #..prevents edge clipping when saving the picture.

# Left Y-axis: C-index (blue)
l1 = ax1.plot(df_c["k"].values, df_c["c_index"].values,
              color="blue", marker="o", label="C-index")
ax1.set_xlabel("Number of top features included", fontsize=14)  # <-- updated
ax1.set_ylabel("C-index", fontsize=14, color="blue")
ax1.tick_params(axis='y', labelcolor="blue")

# Right Y-axis: BIC (green, slightly translucent)
ax2 = ax1.twinx()
l2 = ax2.plot(df_b["k"].values, df_b["BIC"].values,
              color="green", alpha=0.6, marker="^", label="BIC")
ax2.set_ylabel("BIC", fontsize=14, color="green")
ax2.tick_params(axis='y', labelcolor="green")

# Align x-axis to shared k-range (up to 40)
xmin = min(df_c["k"].min(), df_b["k"].min())
ax1.set_xlim(xmin, 40)

# Vertical red dotted line at BIC minimum
vline = ax1.axvline(k_bic_min, color="red", linestyle="--", linewidth=2,
                    label=f"BIC min (k={int(k_bic_min)})")

# Title and legend moved to right-center to avoid covering curves
ax1.set_title("C-index (left) and BIC (right) vs top-k features", fontsize=14)
lines = l1 + l2 + [vline]
labels = [ln.get_label() for ln in lines]
ax1.legend(lines, labels, fontsize=12, loc="center right", bbox_to_anchor=(0.98, 0.5),
           frameon=True)

# Tight bounding box + small padding avoids clipping twin-axis labels/legend
fig.savefig("ablation_BIC_Cindex_Model1.png", dpi=300, bbox_inches="tight", pad_inches=0.08)
#plt.close(fig)
plt.tight_layout()
plt.show()

The minimal optimal list of features given by the BIC criteria is: ['AF_age_4', 'AF_age_3', 'weight', 'AF_age_2', 'bmi', 'DM', 'Marital', 'TIA_CVA_Stroke', 'RR Value 3mo', 'Depression', 'QRS Value 3mo', 'HTN', 'Cognitive_Impairment', 'Sodium Value 3mo', 'Calcium Value 3mo', 'HCT_missing', 'Osteoarthritis', 'race', 'Calcium_missing', 'Insurance', 'Osteoporosis', 'Parkinson']

For Model 1: ['AF_Age', 'weight', 'bmi', 'Depression', 'Marital', 'RR Value 3mo', 'TIA_CVA_Stroke', 'QRS Value 3mo', 'Cognitive_Deficit', 'Creatinine Value 3mo', 'HTN', 'Sodium Value 3mo', 'Osteoarthritis', 'Calcium Value 3mo', 'height', 'Calcium_missing', 'OSA', 'HCT_missing', 'ADI_STATERANK_missing', 'Parkinson']

For Model 2: ['AF_Age', 'bmi', 'weight', 'Depression', 'TIA_CVA_Stroke', 'DM', 'Calcium Value 3mo', 'RR Value 3mo', 'OSA', 'Osteoarthritis', 'Marital', 'HCT_missing']

For Model 3: ['AF_Age', 'weight', 'bmi', 'DM', 'Marital', 'Depression', 'RR Value 3mo', 'TIA_CVA_Stroke', 'QRS Value 3mo', 'Cognitive_Deficit', 'Creatinine Value 3mo', 'Sodium Value 3mo', 'HCT_missing', 'Calcium Value 3mo', 'HTN', 'Osteoarthritis', 'Potassium Value 3mo', 'Calcium_missing', 'PPI', 'height', 'Insurance', 'Parkinson']

For Model 4: ['AF_Age', 'weight', 'bmi', 'DM', 'Depression', 'Marital', 'QRS Value 3mo', 'Creatinine Value 3mo', 'RR Value 3mo', 'Osteoarthritis', 'Calcium_missing', 'Cognitive_Deficit']

#Retrain the model with only the BIC-selected features

(now, test is no longer held-out)

In [ ]:
y_event_train = y_train["event"].astype(int)   # 1 = dementia observed, 0 = censored
y_time_train  = y_train["time"].astype(float)  # follow-up time in years

y_event_val = y_val["event"].astype(int)
y_time_val  = y_val["time"].astype(float)

y_event_test = y_test["event"].astype(int)
y_time_test  = y_test["time"].astype(float)

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sksurv.metrics import concordance_index_censored

# =========================================================
# 1. BIC-restricted features
# =========================================================

#...the following is for Model 3 currently, as currently only Model 3 is supposed to be deployed as the web app.
bic_features = [
    'AF_Age', 'weight', 'bmi', 'DM', 'Marital', 'Depression', 'RR Value 3mo',
    'TIA_CVA_Stroke', 'QRS Value 3mo', 'Cognitive_Deficit', 'Creatinine Value 3mo',
    'Sodium Value 3mo', 'HCT_missing', 'Calcium Value 3mo', 'HTN', 'Osteoarthritis',
    'Potassium Value 3mo', 'Calcium_missing', 'PPI', 'height', 'Insurance', 'Parkinson'
    ]

X_train_bic = X_train.loc[:, bic_features].copy()
X_val_bic   = X_val.loc[:, bic_features].copy()
X_test_bic  = X_test.loc[:, bic_features].copy()

# =========================================================
# 2. XGBoost Cox label encoding
#    event=1 -> +time
#    event=0 -> -time
# =========================================================
def to_xgb_cox_label(y_df, eps=None):
    event = y_df["event"].to_numpy().astype(int)
    time_ = y_df["time"].to_numpy().astype(float)

    if np.any(time_ < 0):
        raise ValueError("Negative follow-up times found.")

    pos = time_[time_ > 0]
    if eps is None:
        eps = min(1e-6, pos.min() / 10.0) if len(pos) else 1e-6

    time_ = np.where(time_ == 0, eps, time_)
    y_signed = np.where(event == 1, time_, -time_)

    return y_signed

def cindex_from_risk(y_df, risk):
    return concordance_index_censored(
        y_df["event"].astype(bool).to_numpy(),
        y_df["time"].astype(float).to_numpy(),
        np.asarray(risk).ravel()
    )[0]

# =========================================================
# 3. DMatrix
# =========================================================
dtrain = xgb.DMatrix(X_train_bic, label=to_xgb_cox_label(y_train), feature_names=bic_features)
dvalid = xgb.DMatrix(X_val_bic,   label=to_xgb_cox_label(y_val),   feature_names=bic_features)
dtest  = xgb.DMatrix(X_test_bic,  label=to_xgb_cox_label(y_test),  feature_names=bic_features)

# =========================================================
# 4. Reuse previous best hyperparameters from metadata
# =========================================================
params = {
    "objective":        "survival:cox",
    "booster":          "gbtree",
    "tree_method":      "gpu_hist",      # switch to "hist" if needed
    "predictor":        "gpu_predictor", # switch to "auto" if needed
    "eval_metric":      "cox-nloglik",
    "verbosity":        0,
    "seed":             42,

    "max_depth":        3,
    "min_child_weight": 3,
    "gamma":            0.0,
    "subsample":        0.8,
    "colsample_bytree": 0.6,
    "eta":              0.05,
    "lambda":           0.0,
    "alpha":            0.1,
    "max_bin":          264,
}

# Optional:
# params["gpu_id"] = 0

# =========================================================
# 5. Refit on BIC subset with early stopping
# =========================================================
model_bic_fixedhp = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=8000,
    evals=[(dvalid, "val")],
    early_stopping_rounds=300,
    verbose_eval=False
)

best_round_bic = model_bic_fixedhp.best_iteration

# =========================================================
# 6. Evaluate
# =========================================================
risk_train = model_bic_fixedhp.predict(dtrain, output_margin=True)
risk_val   = model_bic_fixedhp.predict(dvalid, output_margin=True)
risk_test  = model_bic_fixedhp.predict(dtest,  output_margin=True)

c_train = cindex_from_risk(y_train, risk_train)
c_val   = cindex_from_risk(y_val,   risk_val)
c_test  = cindex_from_risk(y_test,  risk_test)

print("=== XGB Cox on BIC set using previous best hyperparameters ===")
print("Best round:", best_round_bic)
print(f"Train C-index: {c_train:.4f}")
print(f"Val   C-index: {c_val:.4f}")
print(f"Test  C-index: {c_test:.4f}")

# =========================================================
# 7. Optional final refit on TRAIN+VAL using best_round_bic+1
# =========================================================
X_trv = pd.concat([X_train_bic, X_val_bic], axis=0)
y_trv = pd.concat([y_train, y_val], axis=0)

dtrv = xgb.DMatrix(
    X_trv,
    label=to_xgb_cox_label(y_trv),
    feature_names=bic_features
)

model_bic_fixedhp_final = xgb.train(
    params=params,
    dtrain=dtrv,
    num_boost_round=best_round_bic + 1,
    verbose_eval=False
)

risk_test_final = model_bic_fixedhp_final.predict(dtest, output_margin=True)
c_test_final = cindex_from_risk(y_test, risk_test_final)

print(f"Final TEST C-index after refit on TRAIN+VAL: {c_test_final:.4f}")

save_xgb_cox_model(
    model_bic_fixedhp_final,
    prefix="xgb_cox_gpu_bic_prev_best_hparams",
    selected_cols=bic_features,
    best_params=params,
    best_round=best_round_bic,
    extra_meta={
        "selection": "BIC_features_previous_best_hparams",
        "n_features": len(bic_features),
        "source_hparams": "previous_77_feature_model",
        "test_cindex": float(c_test_final)
    }
)

#Generate DCA thresholds and KM curves

you can load the model trained with minimal optimal set of features here.

In [ ]:
import json
import xgboost as xgb

# =========================================================
# 1. Define prefix (MUST match what you used during saving)
# =========================================================
prefix = "xgb_cox_gpu_bic_prev_best_hparams"

# =========================================================
# 2. Load booster
# =========================================================
booster = xgb.Booster()
booster.load_model(f"{prefix}.json")

# =========================================================
# 3. Load metadata
# =========================================================
with open(f"{prefix}_meta.json") as f:
    meta = json.load(f)

feature_names = meta["feature_names"]

print("\nFeature names from meta:")
print(feature_names)

# =========================================================
# 4. Sanity check against booster
# =========================================================
print("\nFeature names from booster:")
print(booster.feature_names)

# STRICT check
assert feature_names == booster.feature_names, "Feature order mismatch!"

print("\n✅ Feature alignment OK")

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, xgboost as xgb
from lifelines import KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts
from typing import Tuple

# ------------ helpers ------------
def align_cols(df, cols): return df.loc[:, cols]

def predict_margin(booster, X):
    return booster.predict(xgb.DMatrix(X), output_margin=True)

def breslow_baseline(y_df, risk):
    df = y_df.copy().reset_index(drop=True)
    order = np.argsort(df["time"].values)
    T = df["time"].values[order]
    E = df["event"].astype(int).values[order]
    r = np.asarray(risk).ravel()[order]

    uniq_t, idx_first = np.unique(T, return_index=True)
    rev_csum = np.cumsum(r[::-1])[::-1]
    dj = np.add.reduceat(E, idx_first)
    denom = rev_csum[idx_first]
    dH = np.where(denom > 0, dj / denom, 0.0)
    H0 = np.cumsum(dH)
    S0 = np.exp(-H0)
    return uniq_t, S0

def baseline_S0_at(uniq_t, S0, t_star):
    if t_star <= uniq_t[0]: return 1.0
    if t_star >= uniq_t[-1]: return float(S0[-1])
    j = np.searchsorted(uniq_t, t_star, side="right") - 1
    return float(S0[j])

def absolute_risk_from_margin(margin, S0_tstar):
    r = np.exp(np.asarray(margin).ravel())
    return 1.0 - (S0_tstar ** r)

def km_censor_survival(y_df):
    km = KaplanMeierFitter()
    km.fit(durations=y_df["time"], event_observed=1 - y_df["event"].astype(int))
    return km

def dca_ipcw(y_df, risk_prob, t_star, thresholds):
    T = y_df["time"].values
    E = y_df["event"].astype(int).values
    U = np.minimum(T, t_star)
    kmc = km_censor_survival(y_df)
    G = np.clip(kmc.predict(U).values, 1e-6, 1.0)   # S_c(U)
    w = 1.0 / G
    D = ((T <= t_star) & (E == 1)).astype(int)
    Nw = w.sum()

    rows = []
    for pt in thresholds:
        pos = (risk_prob >= pt).astype(int)
        TPw = (w * pos * D).sum()
        FPw = (w * pos * (1 - D)).sum()
        NB_model = (TPw / Nw) - (FPw / Nw) * (pt / (1 - pt))
        TPw_all = (w * D).sum()
        FPw_all = (w * (1 - D)).sum()
        NB_all  = (TPw_all / Nw) - (FPw_all / Nw) * (pt / (1 - pt))
        NB_none = 0.0
        rows.append((pt, NB_model, NB_all, NB_none))
    df = pd.DataFrame(rows, columns=["threshold","NB_model","NB_all","NB_none"])
    return df

def pick_threshold_band(dca_df):
    cond = (dca_df["NB_model"] > dca_df["NB_all"]) & (dca_df["NB_model"] > dca_df["NB_none"])
    if not cond.any(): return None
    band = dca_df.loc[cond, "threshold"]
    return float(band.min()), float(band.max())

def plot_dca_enhanced(df: pd.DataFrame, thresholds: Tuple[float,float] | None = None,
                      t_star: float = 5.0, per: int = 100,
                      zoom_ylim: Tuple[float,float] | None = None,
                      figsize=(8,7)):
    """
    Top: standard DCA (NB per patient).
    Bottom: ΔNB = NB_model − max(NB_all, NB_none), scaled per `per` patients.
    """
    thr = df["threshold"].to_numpy()
    nb_m = df["NB_model"].to_numpy()
    nb_a = df["NB_all"].to_numpy()
    nb_n = df["NB_none"].to_numpy()
    delta = nb_m - np.maximum(nb_a, nb_n)

    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=figsize,
                                   gridspec_kw={"height_ratios":[2,1]})

    mask = (nb_m > nb_a) & (nb_m > nb_n)
    ax1.fill_between(thr, nb_m, nb_n, where=mask, alpha=0.15, step="pre", label="Model > all/none")
    ax1.plot(thr, nb_m, linewidth=2.5, marker="o", markersize=3, label="Model")
    ax1.plot(thr, nb_a, linestyle="--", label="Treat all")
    ax1.plot(thr, nb_n, linestyle=":",  label="Treat none")

    if thresholds is not None:
        low, high = thresholds
        for x, lab in [(low, f"low={low:.2f}"), (high, f"high={high:.2f}")]:
            ax1.axvline(x, linestyle="--")
            ax1.text(x, ax1.get_ylim()[1]*0.95, f"  {lab}", va="top")

    if zoom_ylim is not None:
        ax1.set_ylim(*zoom_ylim)

    ax1.set_ylabel("Net benefit (per patient)", fontsize=12)
    ax1.set_title(f"Decision Curve Analysis at t*={t_star}y", fontsize=12)
    ax1.legend(loc="best")

    ax2.axhline(0.0, linewidth=1)
    ax2.fill_between(thr, 0, delta*per, where=delta>0, alpha=0.25, step="pre")
    ax2.plot(thr, delta*per, linewidth=2.0, label="ΔNB = Model − best comparator")
    if thresholds is not None:
        ax2.axvline(low, linestyle="--");  ax2.axvline(high, linestyle="--")
    ax2.set_xlabel("Risk threshold (probability)", fontsize=12)
    ax2.set_ylabel(f"Δ Net benefit (per {per})", fontsize=12)
    plt.tight_layout()
    plt.show()
    return fig, (ax1, ax2)

def group_labels_from_band(abs_risk, band):
    lo, hi = band
    # nearest-integer labels for readability; probabilities kept exact for grouping
    def pct(x): return int(round(100*x))
    s = pd.Series(index=np.arange(len(abs_risk)), dtype="object")
    s[abs_risk <= lo] = f"Low (≤{pct(lo)}%)"
    s[(abs_risk > lo) & (abs_risk <= hi)] = f"Medium ({pct(lo)}–{pct(hi)}%)"
    s[abs_risk > hi] = f"High (≥{pct(hi)}%)"
    return s.astype("category")

def observed_km_risk(y_df, t_star):
    """1 - KM survival at t_star (right-continuous)."""
    km = KaplanMeierFitter()
    km.fit(y_df["time"], y_df["event"])
    # right-continuous step
    t_idx = np.searchsorted(km.survival_function_.index.values, t_star, side="right") - 1
    S = float(km.survival_function_.iloc[max(t_idx,0), 0]) if t_idx >= 0 else 1.0
    return 1.0 - S

def reorder_low_med_high(grp_cat: pd.Series) -> pd.Series:
    """
    Reorder a categorical with labels that start with 'Low', 'Medium', 'High'
    regardless of the numeric cutpoints embedded in the string.
    """
    if not pd.api.types.is_categorical_dtype(grp_cat):
        grp_cat = grp_cat.astype("category")
    cats = list(grp_cat.cat.categories)

    def key(c):
        s = str(c)
        if s.startswith("Low"): return 0
        if s.startswith("Medium"): return 1
        if s.startswith("High"): return 2
        return 3  # anything unexpected goes last

    new_cats = sorted(cats, key=key)
    return grp_cat.cat.reorder_categories(new_cats, ordered=True)

# ------------ main wrapper ------------
def xgb_cox_dca_km_and_table(
    booster, meta,
    X_train, y_train, X_val, y_val, X_test, y_test,
    t_star=5.0, dca_range=(0.02, 0.30), dca_points=80,
    horizon_years=30.0, zoom_ylim=None, per_delta_nb=100,
    gdrive_folder: str | None = None, save_prefix="XGB_Cox"
):
    feats = meta["feature_names"]
    Xtr = align_cols(X_train, feats)
    Xva = align_cols(X_val,   feats)
    Xte = align_cols(X_test,  feats)
    ytr = y_train.reset_index(drop=True)
    yva = y_val.reset_index(drop=True)
    yte = y_test.reset_index(drop=True)

    # TRAIN+VAL for baseline & DCA
    Xtrv = pd.concat([Xtr, Xva], axis=0)
    ytrv = pd.concat([ytr, yva], axis=0).reset_index(drop=True)

    # Baseline survival & absolute risk
    margin_trv = predict_margin(booster, Xtrv)
    uniq_t, S0 = breslow_baseline(ytrv, np.exp(margin_trv))
    S0_tstar   = baseline_S0_at(uniq_t, S0, t_star)

    abs_risk_trv = absolute_risk_from_margin(margin_trv, S0_tstar)
    abs_risk_te  = absolute_risk_from_margin(predict_margin(booster, Xte), S0_tstar)

    # DCA on TRAIN+VAL and enhanced plot
    pts = np.linspace(dca_range[0], dca_range[1], dca_points)
    dca_df = dca_ipcw(ytrv, abs_risk_trv, t_star, pts)
    band = pick_threshold_band(dca_df)
    plot_dca_enhanced(dca_df, thresholds=band, t_star=t_star, per=per_delta_nb, zoom_ylim=zoom_ylim)

    if band is None:
        # fallback to tertiles if no positive region
        qs = np.quantile(abs_risk_trv, [1/3, 2/3])
        band = (float(qs[0]), float(qs[1]))
        print("[WARN] No positive DCA region; using TRAIN+VAL tertiles:", band)

    # Assign TEST risk groups
    grp_te = group_labels_from_band(abs_risk_te, band)

    # PLOT THE KM CURVE. Figure: height = 2x width (adjust width_in as desired)
    width_in = 8
    height_in = 1.4 * width_in

    # Desired display order
    desired_order = ["Low (≤2%)", "Medium (2–7%)", "High (≥7%)"]
    # Keep only those levels that actually exist in grp_te:
    order_present = [lvl for lvl in desired_order if lvl in list(grp_te.cat.categories)]
    # Reorder the categorical to match our desired order
    grp_te = grp_te.cat.reorder_categories(order_present, ordered=True)

    timeline = np.linspace(0, horizon_years, int(horizon_years * 12) + 1)

    fig, ax = plt.subplots(figsize=(width_in, height_in))
    kmf = KaplanMeierFitter()

    # Plot in the forced order
    for lvl in order_present:
        ix = (grp_te == lvl)
        if ix.sum() == 0:
            continue
        kmf.fit(
            yte.loc[ix, "time"],
            yte.loc[ix, "event"],
            label=str(lvl),
            timeline=timeline
        )
        kmf.plot(ax=ax)

    ax.set_xlim(0, horizon_years)
    ax.set_xlabel("Years since AF + 3 months", fontsize=14)
    ax.set_ylabel("Dementia-free survival  S(t)", fontsize=14)
    ax.set_title(f"Dementia-free survival by DCA-derived risk bands (t*={t_star}y)")
    ax.legend(fontsize=13)  # legend will follow plotting order

    # Numbers at risk table every 5 years, in the same forced order
    breaks = np.arange(0, horizon_years + 1, 5)
    kmfs = []
    for lvl in order_present:
        ix = (grp_te == lvl)
        if ix.sum() == 0:
            continue
        kmfs.append(
            KaplanMeierFitter().fit(
                yte.loc[ix, "time"],
                yte.loc[ix, "event"],
                label=str(lvl)
            )
        )

    if kmfs:
        add_at_risk_counts(*kmfs, ax=ax, xticks=breaks, fontsize=12)

    plt.tight_layout()
    fig.savefig(f"{save_prefix}_KM_TEST.png", dpi=300, bbox_inches="tight")
    plt.show()

    # ---- Table: 5-year risk (%) & N per group (TEST) ----
    rows = []
    for lvl in grp_te.cat.categories:
        ix = (grp_te == lvl)
        N = int(ix.sum())
        pred_mean = float(abs_risk_te[ix].mean()) if N>0 else np.nan
        # observed 5y risk via KM
        obs = observed_km_risk(yte.loc[ix], t_star) if N>0 else np.nan
        rows.append({"group": str(lvl),
                     "N": N,
                     "Predicted 5y risk % (mean)": round(100*pred_mean, 2) if pred_mean==pred_mean else np.nan,
                     "Observed 5y risk % (KM)": round(100*obs, 2) if obs==obs else np.nan})
    risk_tbl = pd.DataFrame(rows)

    # Save to Google Drive if path provided
    if gdrive_folder is not None:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            path = gdrive_folder.rstrip("/") + f"/{save_prefix}_5y_risk_by_group_TEST.csv"
            risk_tbl.to_csv(path, index=False)
            print("Saved:", path)
        except Exception as e:
            print("Could not save to Drive:", e)

    return {
        "dca": dca_df, "thresholds": band, "S0_times": uniq_t, "S0": S0, "S0_tstar": S0_tstar,
        "abs_risk_trainval": abs_risk_trv, "abs_risk_test": abs_risk_te,
        "groups_test": grp_te, "risk_table": risk_tbl
    }


In [ ]:
from lifelines import CoxPHFitter

def compute_group_hr_text(
    time,
    event,
    groups,
    ref_token="Low",
    compare_token="High",
    drop_other_groups=True
):
    """
    Compute Cox HR(compare vs ref) when risk groups are labeled with
    human-readable strings like 'Low (<2%)', 'Medium (2-7.3%)',
    'High (>7.3%)', using text token matching.

    Parameters
    ----------
    time : array-like
        Event or censoring times.
    event : array-like
        Event indicator (1 = event, 0 = censored).
    groups : array-like
        Risk group labels (e.g., 'Low (<2%)', 'Medium (2-7.3%)', 'High (>7.3%)').
    ref_token : str, default="Low"
        Text token used to identify the reference group (e.g. "Low").
    compare_token : str, default="High"
        Text token used to identify the comparison group (e.g. "High").
    drop_other_groups : bool, default=True
        If True, restrict analysis to only ref_token and compare_token.
        If False, other groups can stay, but only the binary contrast
        compare vs ref is modeled via a 0/1 indicator.

    Returns
    -------
    results : dict
        {
          'HR': float,
          'lower_95': float,
          'upper_95': float,
          'coef': float,
          'se': float,
          'summary': pd.DataFrame,
          'cph_model': CoxPHFitter
        }
    """

    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    groups = np.asarray(groups)

    # Basic mask for finite values
    mask = np.isfinite(time) & np.isfinite(event) & pd.notnull(groups)
    time = time[mask]
    event = event[mask]
    groups = groups[mask]

    if time.size == 0:
        raise ValueError("After removing NaNs, no observations remain.")

    # --- Normalize group labels using text tokens ---
    norm = []
    for g in groups:
        s = str(g)
        s_lower = s.lower()
        if ref_token.lower() in s_lower:
            norm.append(ref_token)      # e.g., "Low"
        elif compare_token.lower() in s_lower:
            norm.append(compare_token)  # e.g., "High"
        elif "medium" in s_lower:       # in case you want Medium too
            norm.append("Medium")
        else:
            norm.append(None)           # Unrecognized
    norm = np.array(norm, dtype=object)

    # Optional: restrict to ref vs compare only
    if drop_other_groups:
        keep = (norm == ref_token) | (norm == compare_token)
        time = time[keep]
        event = event[keep]
        norm = norm[keep]

    if time.size == 0:
        raise ValueError(
            f"No observations found for ref_token='{ref_token}' and "
            f"compare_token='{compare_token}'. "
            f"Check that your group labels contain these tokens."
        )

    unique_norm, counts = np.unique(norm, return_counts=True)
    uniq_dict = dict(zip(unique_norm.tolist(), counts.tolist()))

    if ref_token not in uniq_dict:
        raise ValueError(
            f"Reference token '{ref_token}' not present in data after filtering. "
            f"Available normalized labels: {uniq_dict}"
        )
    if compare_token not in uniq_dict:
        raise ValueError(
            f"Compare token '{compare_token}' not present in data after filtering. "
            f"Available normalized labels: {uniq_dict}"
        )

    # Build analysis DataFrame
    df = pd.DataFrame({
        "time": time,
        "event": event
    })

    # Binary indicator: 1 = compare (e.g., "High"), 0 = ref (e.g., "Low")
    df["grp_bin"] = (norm == compare_token).astype(int)

    # Fit Cox model
    cph = CoxPHFitter()
    cph.fit(df, duration_col="time", event_col="event", formula="grp_bin")

    row = cph.summary.loc["grp_bin"]
    coef = row["coef"]
    se = row["se(coef)"]
    HR = np.exp(coef)
    lower_95 = np.exp(coef - 1.96 * se)
    upper_95 = np.exp(coef + 1.96 * se)

    return {
        "HR": HR,
        "lower_95": lower_95,
        "upper_95": upper_95,
        "coef": coef,
        "se": se,
        "summary": cph.summary,
        "cph_model": cph
    }


In [ ]:
from lifelines import CoxPHFitter
from scipy.stats import norm

def compute_royston_sauerbrei_D(time, event, lp):
    """
    Compute Royston–Sauerbrei D-statistic and its HR-equivalent (D-index)
    from a prognostic index (lp = linear predictor) and survival data.

    Parameters
    ----------
    time : array-like
        Event or censoring times.
    event : array-like
        Event indicator (1 = event, 0 = censored).
    lp : array-like
        Prognostic index (e.g., linear predictor/log hazard).

    Returns
    -------
    results : dict
        {
          'D': float,
          'D_index': float,        # exp(D)
          'se': float,             # standard error of D
          'lower': float,          # 95% CI lower for D
          'upper': float,          # 95% CI upper for D
          'cox_model': CoxPHFitter
        }
    """
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    lp = np.asarray(lp, dtype=float)

    # Drop any NaNs just in case
    mask = np.isfinite(time) & np.isfinite(event) & np.isfinite(lp)
    time = time[mask]
    event = event[mask]
    lp = lp[mask]

    n = len(lp)
    if n < 5:
        raise ValueError("Not enough observations to compute D (n < 5).")

    # Order by lp
    order = np.argsort(lp)
    time_sorted = time[order]
    event_sorted = event[order]

    # Expected standard normal order statistics (rankits)
    # Common approximation: (i - 0.375) / (n + 0.25)
    i = np.arange(1, n + 1)
    probs = (i - 0.375) / (n + 0.25)
    z = norm.ppf(probs)

    # Scale rankits by kappa = sqrt(8/pi)  (Royston & Sauerbrei)
    kappa = np.sqrt(8.0 / np.pi)
    z_scaled = z * kappa

    df = pd.DataFrame({
        "time": time_sorted,
        "event": event_sorted,
        "z": z_scaled
    })

    cph = CoxPHFitter()
    cph.fit(df, duration_col="time", event_col="event", formula="z")

    beta = float(cph.params_["z"])                # this is D
    se = float(cph.standard_errors_["z"])
    D = beta
    D_index = np.exp(D)                          # HR equivalent
    lower = D - 1.96 * se
    upper = D + 1.96 * se

    return {
        "D": D,
        "D_index": D_index,
        "se": se,
        "lower": lower,
        "upper": upper,
        "cox_model": cph
    }


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, xgboost as xgb
from lifelines import KaplanMeierFitter
from lifelines import CoxPHFitter
from lifelines.plotting import add_at_risk_counts
from typing import Tuple

# ------------ helpers ------------
def align_cols(df, cols):
    return df.loc[:, cols]

def predict_margin(booster, X):
    return booster.predict(xgb.DMatrix(X), output_margin=True)

def breslow_baseline(y_df, risk):
    df = y_df.copy().reset_index(drop=True)
    order = np.argsort(df["time"].values)
    T = df["time"].values[order]
    E = df["event"].astype(int).values[order]
    r = np.asarray(risk).ravel()[order]

    uniq_t, idx_first = np.unique(T, return_index=True)
    rev_csum = np.cumsum(r[::-1])[::-1]
    dj = np.add.reduceat(E, idx_first)
    denom = rev_csum[idx_first]
    dH = np.where(denom > 0, dj / denom, 0.0)
    H0 = np.cumsum(dH)
    S0 = np.exp(-H0)
    return uniq_t, S0

def baseline_S0_at(uniq_t, S0, t_star):
    if t_star <= uniq_t[0]: return 1.0
    if t_star >= uniq_t[-1]: return float(S0[-1])
    j = np.searchsorted(uniq_t, t_star, side="right") - 1
    return float(S0[j])

def absolute_risk_from_margin(margin, S0_tstar):
    r = np.exp(np.asarray(margin).ravel())
    return 1.0 - (S0_tstar ** r)

def km_censor_survival(y_df):
    km = KaplanMeierFitter()
    km.fit(durations=y_df["time"], event_observed=1 - y_df["event"].astype(int))
    return km

def dca_ipcw(y_df, risk_prob, t_star, thresholds):
    T = y_df["time"].values
    E = y_df["event"].astype(int).values
    U = np.minimum(T, t_star)
    kmc = km_censor_survival(y_df)
    G = np.clip(kmc.predict(U).values, 1e-6, 1.0)   # S_c(U)
    w = 1.0 / G
    D = ((T <= t_star) & (E == 1)).astype(int)
    Nw = w.sum()

    rows = []
    for pt in thresholds:
        pos = (risk_prob >= pt).astype(int)
        TPw = (w * pos * D).sum()
        FPw = (w * pos * (1 - D)).sum()
        NB_model = (TPw / Nw) - (FPw / Nw) * (pt / (1 - pt))
        TPw_all = (w * D).sum()
        FPw_all = (w * (1 - D)).sum()
        NB_all  = (TPw_all / Nw) - (FPw_all / Nw) * (pt / (1 - pt))
        NB_none = 0.0
        rows.append((pt, NB_model, NB_all, NB_none))
    df = pd.DataFrame(rows, columns=["threshold","NB_model","NB_all","NB_none"])
    return df

def pick_threshold_band(dca_df):
    cond = (dca_df["NB_model"] > dca_df["NB_all"]) & (dca_df["NB_model"] > dca_df["NB_none"])
    if not cond.any(): return None
    band = dca_df.loc[cond, "threshold"]
    return float(band.min()), float(band.max())

def plot_dca_enhanced(df: pd.DataFrame, thresholds: Tuple[float,float] | None = None,
                      t_star: float = 5.0, per: int = 100,
                      zoom_ylim: Tuple[float,float] | None = None,
                      figsize=(8,7)):
    """
    Top: standard DCA (NB per patient).
    Bottom: ΔNB = NB_model − max(NB_all, NB_none), scaled per `per` patients.
    """
    thr = df["threshold"].to_numpy()
    nb_m = df["NB_model"].to_numpy()
    nb_a = df["NB_all"].to_numpy()
    nb_n = df["NB_none"].to_numpy()
    delta = nb_m - np.maximum(nb_a, nb_n)

    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=figsize,
                                   gridspec_kw={"height_ratios":[2,1]})

    mask = (nb_m > nb_a) & (nb_m > nb_n)
    ax1.fill_between(thr, nb_m, nb_n, where=mask, alpha=0.15, step="pre", label="Model > all/none")
    ax1.plot(thr, nb_m, linewidth=2.5, marker="o", markersize=3, label="Model")
    ax1.plot(thr, nb_a, linestyle="--", label="Treat all")
    ax1.plot(thr, nb_n, linestyle=":",  label="Treat none")

    if thresholds is not None:
        low, high = thresholds
        for x, lab in [(low, f"low={low:.2f}"), (high, f"high={high:.2f}")]:
            ax1.axvline(x, linestyle="--")
            ax1.text(x, ax1.get_ylim()[1]*0.95, f"  {lab}", va="top")

    if zoom_ylim is not None:
        ax1.set_ylim(*zoom_ylim)

    ax1.set_ylabel("Net benefit (per patient)", fontsize=12)
    ax1.set_title(f"Decision Curve Analysis at t*={t_star}y", fontsize=12)
    ax1.legend(loc="best")

    ax2.axhline(0.0, linewidth=1)
    ax2.fill_between(thr, 0, delta*per, where=delta>0, alpha=0.25, step="pre")
    ax2.plot(thr, delta*per, linewidth=2.0, label="ΔNB = Model − best comparator")
    if thresholds is not None:
        ax2.axvline(low, linestyle="--");  ax2.axvline(high, linestyle="--")
    ax2.set_xlabel("Risk threshold (probability)", fontsize=12)
    ax2.set_ylabel(f"Δ Net benefit (per {per})", fontsize=12)
    plt.tight_layout()
    plt.show()
    return fig, (ax1, ax2)

def group_labels_from_band(abs_risk, band):
    lo, hi = band
    # friendly percent strings (e.g., "0.5", "2", "7")
    def pct_str(x):
        s = f"{100*x:.1f}".rstrip("0").rstrip(".")
        return s
    s = pd.Series(index=np.arange(len(abs_risk)), dtype="object")
    s[abs_risk <= lo] = f"Low (≤{pct_str(lo)}%)"
    s[(abs_risk > lo) & (abs_risk <= hi)] = f"Medium ({pct_str(lo)}–{pct_str(hi)}%)"
    s[abs_risk > hi] = f"High (≥{pct_str(hi)}%)"
    return s.astype("category")

def observed_km_risk(y_df, t_star):
    """1 - KM survival at t_star (right-continuous)."""
    km = KaplanMeierFitter()
    km.fit(y_df["time"], y_df["event"])
    t_idx = np.searchsorted(km.survival_function_.index.values, t_star, side="right") - 1
    S = float(km.survival_function_.iloc[max(t_idx,0), 0]) if t_idx >= 0 else 1.0
    return 1.0 - S

def reorder_low_med_high(grp_cat: pd.Series) -> pd.Series:
    """
    Reorder a categorical with labels that start with 'Low', 'Medium', 'High'
    regardless of the numeric cutpoints embedded in the string.
    """
    if not pd.api.types.is_categorical_dtype(grp_cat):
        grp_cat = grp_cat.astype("category")
    cats = list(grp_cat.cat.categories)
    def key(c):
        s = str(c)
        if s.startswith("Low"): return 0
        if s.startswith("Medium"): return 1
        if s.startswith("High"): return 2
        return 3
    new_cats = sorted(cats, key=key)
    return grp_cat.cat.reorder_categories(new_cats, ordered=True)

# ------------ main wrapper ------------
def xgb_cox_dca_km_and_table(
    booster, meta,
    X_train, y_train, X_val, y_val, X_test, y_test,
    t_star=5.0, dca_range=(0.02, 0.30), dca_points=80,
    horizon_years=30.0, zoom_ylim=None, per_delta_nb=100,
    gdrive_folder: str | None = None, save_prefix="XGB_Cox"
):
    feats = meta["feature_names"]
    Xtr = align_cols(X_train, feats)
    Xva = align_cols(X_val,   feats)
    Xte = align_cols(X_test,  feats)
    ytr = y_train.reset_index(drop=True)
    yva = y_val.reset_index(drop=True)
    yte = y_test.reset_index(drop=True)

    # TRAIN+VAL for baseline & DCA
    Xtrv = pd.concat([Xtr, Xva], axis=0)
    ytrv = pd.concat([ytr, yva], axis=0).reset_index(drop=True)

    # Baseline survival & absolute risk
    margin_trv = predict_margin(booster, Xtrv)
    uniq_t, S0 = breslow_baseline(ytrv, np.exp(margin_trv))
    S0_tstar   = baseline_S0_at(uniq_t, S0, t_star)

    abs_risk_trv = absolute_risk_from_margin(margin_trv, S0_tstar)
    abs_risk_te  = absolute_risk_from_margin(predict_margin(booster, Xte), S0_tstar)

    # DCA on TRAIN+VAL and enhanced plot
    pts = np.linspace(dca_range[0], dca_range[1], dca_points)
    dca_df = dca_ipcw(ytrv, abs_risk_trv, t_star, pts)
    band = pick_threshold_band(dca_df)
    plot_dca_enhanced(dca_df, thresholds=band, t_star=t_star, per=per_delta_nb, zoom_ylim=zoom_ylim)

    if band is None:
        # fallback to tertiles if no positive region
        qs = np.quantile(abs_risk_trv, [1/3, 2/3])
        band = (float(qs[0]), float(qs[1]))
        print("[WARN] No positive DCA region; using TRAIN+VAL tertiles:", band)

    # Assign TEST risk groups and enforce stable L-M-H ordering
    grp_te = group_labels_from_band(abs_risk_te, band)
    grp_te = reorder_low_med_high(grp_te)
    cats = list(grp_te.cat.categories)  # use actual categories everywhere below

    # PLOT THE KM CURVE
    width_in = 8
    height_in = 1.4 * width_in
    timeline = np.linspace(0, horizon_years, int(horizon_years * 12) + 1)

    fig, ax = plt.subplots(figsize=(width_in, height_in))
    kmf = KaplanMeierFitter()
    for lvl in cats:
        ix = (grp_te == lvl)
        if ix.sum() == 0:
            continue
        kmf.fit(
            yte.loc[ix, "time"],
            yte.loc[ix, "event"],
            label=str(lvl),
            timeline=timeline
        )
        kmf.plot(ax=ax)

    ax.set_xlim(0, horizon_years)
    ax.set_xlabel("Years since AF + 1 year", fontsize=14)
    ax.set_ylabel("Dementia-free survival  S(t)", fontsize=14)
    ax.set_title(f"Dementia-free survival by DCA-derived risk bands (t*={t_star}y)")
    ax.legend(fontsize=13)

    # Numbers at risk table every 5 years, in the same L-M-H order
    breaks = np.arange(0, horizon_years + 1, 5)
    kmfs = []
    for lvl in cats:
        ix = (grp_te == lvl)
        if ix.sum() == 0:
            continue
        kmfs.append(
            KaplanMeierFitter().fit(
                yte.loc[ix, "time"],
                yte.loc[ix, "event"],
                label=str(lvl)
            )
        )

    if kmfs:
        add_at_risk_counts(*kmfs, ax=ax, xticks=breaks, fontsize=12)

    plt.tight_layout()
    fig.savefig(f"{save_prefix}_KM_TEST.png", dpi=300, bbox_inches="tight")
    plt.show()

    # ======== NEW: Royston–Sauerbrei D on TEST ========
    # Linear predictor (log hazard) on TEST
    lp_te = predict_margin(booster, Xte)  # booster and Xte defined earlier in the function

    D_res = compute_royston_sauerbrei_D(
        time=yte["time"].values,
        event=yte["event"].values.astype(int),
        lp=lp_te
    )

    print(f"Royston–Sauerbrei D on TEST: {D_res['D']:.3f} "
          f"(SE {D_res['se']:.3f}, HR-equivalent = {D_res['D_index']:.2f})")

    # ===== NEW: empirical HR high vs low based on TEST risk groups =====
    risk_groups_test = grp_te

    # Optional: sanity check
    unique_labels, counts = np.unique(risk_groups_test, return_counts=True)
    print("Unique TEST risk groups:", list(zip(unique_labels, counts)))

    # Empirical HR: High vs Low on TEST, based on your cutpoints
    hr_res = compute_group_hr_text(
        time=yte["time"].values,
        event=yte["event"].values,
        groups=risk_groups_test,
        ref_token="Low",
        compare_token="High",
        drop_other_groups=True
    )

    print(
        f"HR (High vs Low risk) on TEST: "
        f"{hr_res['HR']:.2f} "
        f"(95% CI {hr_res['lower_95']:.2f}–{hr_res['upper_95']:.2f})"
    )


    # ===== NEW: HR per 1-SD increase in model score on TEST =====
    lp_te = np.asarray(lp_te).ravel()

    lp_te_sd = lp_te.std(ddof=1)
    if lp_te_sd == 0:
        raise ValueError("TEST linear predictor has zero SD; cannot compute per-SD HR.")

    lp_te_z = (lp_te - lp_te.mean()) / lp_te_sd

    df_lp_sd = pd.DataFrame({
        "time": yte["time"].values.astype(float),
        "event": yte["event"].values.astype(int),
        "lp_z": lp_te_z
    })

    cph_lp_sd = CoxPHFitter()
    cph_lp_sd.fit(df_lp_sd, duration_col="time", event_col="event")

    beta_sd = float(cph_lp_sd.params_["lp_z"])
    hr_per_sd = float(np.exp(beta_sd))

    ci_log_sd = cph_lp_sd.confidence_intervals_.loc["lp_z"]
    ci_lower_sd = float(np.exp(ci_log_sd.iloc[0]))
    ci_upper_sd = float(np.exp(ci_log_sd.iloc[1]))
    p_value_sd = float(cph_lp_sd.summary.loc["lp_z", "p"])

    print(
        f"HR per 1-SD increase in model score on TEST: "
        f"{hr_per_sd:.2f} (95% CI {ci_lower_sd:.2f}–{ci_upper_sd:.2f}), "
        f"p={p_value_sd:.3e}"
    )

    # ---- Table: 5-year risk (%) & N per group (TEST) ----
    rows = []
    for lvl in cats:
        ix = (grp_te == lvl)
        N = int(ix.sum())
        pred_mean = float(abs_risk_te[ix].mean()) if N>0 else np.nan
        # observed 5y risk via KM
        obs = observed_km_risk(yte.loc[ix], t_star) if N>0 else np.nan
        rows.append({"group": str(lvl),
                     "N": N,
                     "Predicted 5y risk % (mean)": round(100*pred_mean, 2) if pred_mean==pred_mean else np.nan,
                     "Observed 5y risk % (KM)": round(100*obs, 2) if obs==obs else np.nan})
    risk_tbl = pd.DataFrame(rows)

    # Save to Google Drive if path provided
    if gdrive_folder is not None:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            path = gdrive_folder.rstrip("/") + f"/{save_prefix}_5y_risk_by_group_TEST.csv"
            risk_tbl.to_csv(path, index=False)
            print("Saved:", path)
        except Exception as e:
            print("Could not save to Drive:", e)

    #return {
    #    "dca": dca_df, "thresholds": band, "S0_times": uniq_t, "S0": S0, "S0_tstar": S0_tstar,
    #    "abs_risk_trainval": abs_risk_trv, "abs_risk_test": abs_risk_te,
    #    "groups_test": grp_te, "risk_table": risk_tbl
    #}
    return {
        "dca": dca_df,
        "thresholds": band,
        "S0_times": uniq_t,
        "S0": S0,
        "S0_tstar": S0_tstar,
        "abs_risk_trainval": abs_risk_trv,
        "abs_risk_test": abs_risk_te,
        "groups_test": risk_groups_test,
        "risk_table": risk_tbl,
        # --- previously added D-statistic pieces, if you kept them ---
        "D_test": D_res["D"],
        "D_index_test": D_res["D_index"],
        # -----------------------------------
        "HR_high_vs_low_test": hr_res["HR"],
        "HR_high_vs_low_ci": (hr_res["lower_95"], hr_res["upper_95"]),
        "HR_high_vs_low_details": hr_res,
        "HR_per_sd_test": hr_per_sd,
        "HR_per_sd_test_ci": (ci_lower_sd, ci_upper_sd),
        "HR_per_sd_test_p": p_value_sd,
    }

In [ ]:
def risk_table_at_new_horizon_from_existing_out(out, y_test, new_t_star=2.0):
    """
    Uses the existing 5-year risk bands in out["groups_test"],
    but computes predicted and observed risk at a new horizon, e.g. 2 years.
    """

    # Same TEST groups already defined from 5-year risk bands
    groups = out["groups_test"]

    # Recover each patient's relative risk from their 5-year absolute risk:
    # risk_5y = 1 - S0_5^exp(lp)
    risk_5y = np.asarray(out["abs_risk_test"]).ravel()
    S0_5y = float(out["S0_tstar"])

    eps = 1e-12
    risk_5y = np.clip(risk_5y, eps, 1 - eps)
    S0_5y = np.clip(S0_5y, eps, 1 - eps)

    rel_risk = np.log(1 - risk_5y) / np.log(S0_5y)

    # Get baseline survival at new horizon
    S0_new = baseline_S0_at(
        out["S0_times"],
        out["S0"],
        new_t_star
    )

    # Predicted risk at new horizon
    pred_risk_new = 1 - (S0_new ** rel_risk)

    # Build table
    yte = y_test.reset_index(drop=True)

    rows = []
    cats = list(groups.cat.categories)

    for lvl in cats:
        ix = (groups == lvl).values
        N = int(ix.sum())

        pred_mean = float(pred_risk_new[ix].mean()) if N > 0 else np.nan
        obs = observed_km_risk(yte.loc[ix], new_t_star) if N > 0 else np.nan

        rows.append({
            "group": str(lvl),
            "N": N,
            f"Predicted {new_t_star:g}y risk % (mean)": round(100 * pred_mean, 2),
            f"Observed {new_t_star:g}y risk % (KM)": round(100 * obs, 2)
        })

    return pd.DataFrame(rows)

In [ ]:
#...you can run this with the original 80 variable model (booster=model_final, meta=meta)
# or with the minimal-optimal model (booster=booster, meta=meta)
out = xgb_cox_dca_km_and_table(
    booster=model_final, meta=meta,        #...*** change to model_final if working with whole feature set, booster if working with BIC selected feature set
    X_train=X_train, y_train=y_train,  #...change to X_train_bic if running with the BIC selected model
    X_val=X_val,     y_val=y_val,
    X_test=X_test,   y_test=y_test,
    t_star=5.0,
    dca_range=(0.02, 0.30), dca_points=80,
    horizon_years=22.0,
    zoom_ylim=(-0.02, 0.04),       # optional zoom for the top panel
    per_delta_nb=100,              # ΔNB per 100 patients
    gdrive_folder="/content",
    save_prefix="XGB_Cox_model3_for3_githubpush_rerun"   #...***change the model name and number here.
)

# The 5-year risk table:
out["risk_table"]


In [ ]:
risk_table_2y = risk_table_at_new_horizon_from_existing_out(
    out=out,
    y_test=y_test,
    new_t_star=2.0
)

risk_table_2y

In [ ]:
risk_table_3y = risk_table_at_new_horizon_from_existing_out(
    out=out,
    y_test=y_test,
    new_t_star=3.0
)

risk_table_3y

In [ ]:
#...get hazard ratio between high and low risk groups.
print(out["HR_high_vs_low_test"])
print(out["HR_high_vs_low_ci"])

In [ ]:
print("D on TEST:", out["D_test"], "HR-equivalent:", out["D_index_test"])

In [ ]:
print(out["HR_per_sd_test"])
print(out["HR_per_sd_test_ci"])
print(out["HR_per_sd_test_p"])

In [ ]:
# =========================================================
# Calibration plot at t* = 5y — 3 DCA risk groups (Stanford test cohort)
# The 3 points are exactly the Low / Intermediate / High rows of the risk table
# =========================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

T_STAR = 5.0
lo, hi = out["thresholds"]          # (0.02, 0.052) for the 80-var model

risk = np.asarray(out["abs_risk_test"], dtype=float).ravel()
yte  = y_test.reset_index(drop=True)
time = yte["time"].astype(float).values
event = yte["event"].astype(int).values

def observed_km_risk(t_arr, e_arr, t_star):
    km = KaplanMeierFitter().fit(t_arr, e_arr)
    idx = np.searchsorted(km.survival_function_.index.values, t_star, side="right") - 1
    S = float(km.survival_function_.iloc[max(idx, 0), 0]) if idx >= 0 else 1.0
    return 1.0 - S

# assign the 3 DCA groups
grp = np.where(risk <= lo, 0, np.where(risk <= hi, 1, 2))
labels = [f"Low (<{100*lo:g}%)", f"Intermediate ({100*lo:g}\u2013{100*hi:g}%)", f"High (>{100*hi:g}%)"]

pred_pts, obs_pts, rows = [], [], []
for g in [0, 1, 2]:
    m = grp == g
    p = 100 * risk[m].mean()
    o = 100 * observed_km_risk(time[m], event[m], T_STAR)
    pred_pts.append(p); obs_pts.append(o)
    rows.append({"Risk group": labels[g], "N": int(m.sum()),
                 "Predicted_5y_%": round(p, 2), "Observed_5y_%": round(o, 2)})
pred_pts, obs_pts = np.array(pred_pts), np.array(obs_pts)

# plot
lim = max(pred_pts.max(), obs_pts.max()) * 1.15
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, lim], [0, lim], ls="--", color="grey", label="Perfect calibration")
ax.plot(pred_pts, obs_pts, "o-", color="#2E5AAC", markersize=9, label="Stanford (DCA risk groups)")
for p, o, lab in zip(pred_pts, obs_pts, ["Low", "Intermediate", "High"]):
    ax.annotate(lab, (p, o), textcoords="offset points", xytext=(8, -4), fontsize=10)
ax.set_xlabel(f"Predicted {T_STAR:g}y risk (%)")
ax.set_ylabel(f"Observed {T_STAR:g}y risk (%, KM)")
ax.set_title(f"Calibration at {T_STAR:g} years (Stanford)")
ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig(os.path.join(dev_path, "stanford_calibration_5y_3groups.png"), dpi=300, bbox_inches="tight")
plt.show()

print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# =========================================================
# Calibration plot (decile) at t* = 5y  — Stanford test cohort
# mean predicted risk vs KM-observed risk, per predicted-risk decile
# =========================================================
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

T_STAR = 5.0
N_BINS = 10            # 10 = deciles (the smooth curve); use 5 for quintiles, etc.

# frozen model's 5-year absolute risk + outcome (already computed upstream)
risk = np.asarray(out["abs_risk_test"], dtype=float).ravel()
yte  = y_test.reset_index(drop=True)
time = yte["time"].astype(float).values
event = yte["event"].astype(int).values

def observed_km_risk(t_arr, e_arr, t_star):
    """1 - S(t_star) by Kaplan-Meier within a group (censoring-aware)."""
    km = KaplanMeierFitter().fit(t_arr, e_arr)
    idx = np.searchsorted(km.survival_function_.index.values, t_star, side="right") - 1
    S = float(km.survival_function_.iloc[max(idx, 0), 0]) if idx >= 0 else 1.0
    return 1.0 - S

# bin by quantiles of predicted risk (equal-sized groups)
bins = pd.qcut(risk, N_BINS, labels=False, duplicates="drop")

pred_pts, obs_pts = [], []
for b in np.unique(bins):
    m = bins == b
    pred_pts.append(100 * risk[m].mean())
    obs_pts.append(100 * observed_km_risk(time[m], event[m], T_STAR))
pred_pts, obs_pts = np.array(pred_pts), np.array(obs_pts)

# plot
lim = max(pred_pts.max(), obs_pts.max()) * 1.1
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, lim], [0, lim], ls="--", color="grey", label="Perfect calibration")
ax.plot(pred_pts, obs_pts, "o-", color="#2E5AAC", label=f"Stanford ({N_BINS} bins)")
ax.set_xlabel(f"Predicted {T_STAR:g}y risk (%)")
ax.set_ylabel(f"Observed {T_STAR:g}y risk (%, KM)")
ax.set_title(f"Calibration at {T_STAR:g} years (Stanford)")
ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(dev_path, "stanford_calibration_5y.png"), dpi=300, bbox_inches="tight")
plt.show()

# also print the underlying numbers (like your Extended Data Table, but per bin)
cal_tbl = pd.DataFrame({
    "bin": np.arange(1, len(pred_pts) + 1),
    "n": [int((bins == b).sum()) for b in np.unique(bins)],
    "predicted_5y_%": np.round(pred_pts, 2),
    "observed_5y_%": np.round(obs_pts, 2),
})
print(cal_tbl.to_string(index=False))

In [ ]:
# ==== Export artifacts for the Streamlit bedside calculator ====
import json, numpy as np, pandas as pd, xgboost as xgb

booster = booster   #...this is the 22-feature reduced XGBoost model.
feats = meta["feature_names"]

# TRAIN+VAL matrix in model order (for imputation stats & SHAP background)
Xtrv = pd.concat([X_train.loc[:, feats], X_val.loc[:, feats]], axis=0)

# Mean imputation values from TRAIN+VAL
feature_means = Xtrv.mean(numeric_only=True).to_dict()

# Feature importance order via SHAP (fallback to training order if SHAP not available)
try:
    import shap
    bg = Xtrv.sample(n=min(4000, len(Xtrv)), random_state=1).fillna(pd.Series(feature_means))
    explainer = shap.TreeExplainer(booster, feature_perturbation="interventional", model_output="raw")
    shap_vals = explainer.shap_values(bg.values.astype(np.float32))
    imp = pd.Series(np.abs(shap_vals).mean(axis=0), index=Xtrv.columns).sort_values(ascending=False)
    feature_order = imp.index.tolist()
except Exception:
    feature_order = feats  # fallback

# Baseline & DCA band from your wrapper output
t_star   = 5.0
S0_tstar = float(out["S0_tstar"])
band     = out["thresholds"]  # tuple (low, high) or None

# ---- write artifacts ----
booster.save_model("xgb_cox_model_model3_new.json")
with open("meta_model3_new.json","w") as f: json.dump({"feature_names": feats}, f)
with open("feature_means_model3_new.json","w") as f: json.dump(feature_means, f)
with open("baseline_model3_new.json","w") as f: json.dump({"t_star": t_star, "S0_tstar": S0_tstar}, f)
with open("feature_order_model3_new.json","w") as f: json.dump({"order": feature_order}, f)
with open("dca_band_model3_new.json","w") as f: json.dump({"band": band}, f)

print("Saved artifacts: xgb_cox_model.json, meta.json, feature_means.json, baseline.json, feature_order.json, dca_band.json")

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from lifelines import CoxPHFitter
from sksurv.metrics import concordance_index_censored

# =========================================================
# Killer result:
# Does the XGB model score add prognostic value beyond AF_Age?
# =========================================================

# Assumes you already have:
#   bst            -> trained xgboost.Booster
#   X_test, y_test -> DataFrames
#   meta           -> dict with "feature_names"
#
# y_test must contain:
#   y_test["time"]
#   y_test["event"]

# -----------------------------
# 1) Align test features
# -----------------------------
feature_names = meta.get("feature_names", None)
if feature_names is None:
    raise ValueError("No feature list found in meta['feature_names'].")

missing = [c for c in feature_names if c not in X_test.columns]
if missing:
    raise ValueError(f"Missing columns in X_test: {missing}")

Xte = X_test.loc[:, feature_names].copy()

if "AF_Age" not in Xte.columns:
    raise ValueError("AF_Age not found in X_test / feature set.")

# -----------------------------
# 2) Compute XGB model score on TEST
#    raw margin = linear predictor = log-hazard score
# -----------------------------
dte = xgb.DMatrix(Xte)
risk_test = np.asarray(bst.predict(dte, output_margin=True)).ravel()

# Optional sanity check: C-index of model score alone
c_loaded = concordance_index_censored(
    y_test["event"].astype(bool).values,
    y_test["time"].astype(float).values,
    risk_test
)[0]
print(f"[SANITY] Loaded model TEST C-index = {c_loaded:.6f}")

# -----------------------------
# 3) Standardize AF_Age and model score to 1 SD
#    so HRs are directly interpretable as per-SD HR
# -----------------------------
af_age = Xte["AF_Age"].astype(float).to_numpy()

af_age_sd = af_age.std(ddof=1)
score_sd = risk_test.std(ddof=1)

if af_age_sd == 0:
    raise ValueError("AF_Age has zero SD on TEST.")
if score_sd == 0:
    raise ValueError("Model score has zero SD on TEST.")

af_age_z = (af_age - af_age.mean()) / af_age_sd
score_z  = (risk_test - risk_test.mean()) / score_sd

# -----------------------------
# 4) Build Cox dataframe
# -----------------------------
df_cox = pd.DataFrame({
    "time": y_test["time"].astype(float).values,
    "event": y_test["event"].astype(int).values,
    "AF_Age_z": af_age_z,
    "ModelScore_z": score_z,
})

# Drop any problematic rows just in case
df_cox = df_cox.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

# -----------------------------
# 5) Fit multivariable Cox model
# -----------------------------
cph = CoxPHFitter()
cph.fit(df_cox, duration_col="time", event_col="event")

# -----------------------------
# 6) Extract adjusted HRs per SD with 95% CI
# -----------------------------
summary = cph.summary.copy()

results = []
for var, pretty in [("AF_Age_z", "AF age (per SD)"),
                    ("ModelScore_z", "MIND-AF score (per SD)")]:
    beta = float(summary.loc[var, "coef"])
    hr = float(np.exp(beta))
    ci_lower = float(summary.loc[var, "exp(coef) lower 95%"])
    ci_upper = float(summary.loc[var, "exp(coef) upper 95%"])
    p_value = float(summary.loc[var, "p"])

    results.append({
        "Variable": pretty,
        "Adjusted HR per SD": hr,
        "CI lower 95%": ci_lower,
        "CI upper 95%": ci_upper,
        "p-value": p_value
    })

results_df = pd.DataFrame(results)

print("\nAdjusted Cox model: outcome ~ AF_Age + MIND-AF model score")
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# -----------------------------
# 7) Optional: print full Cox summary
# -----------------------------
print("\nFull CoxPH summary:")
print(
    cph.summary[
        ["coef", "exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]
    ].to_string(float_format=lambda x: f"{x:.4f}")
)

# -----------------------------
# 8) Optional: likelihood ratio style comparison
#    Compare age-only vs age+model
# -----------------------------
df_age_only = df_cox[["time", "event", "AF_Age_z"]].copy()
cph_age = CoxPHFitter()
cph_age.fit(df_age_only, duration_col="time", event_col="event")

print("\nModel comparison:")
print(f"Age-only partial log-likelihood:      {cph_age.log_likelihood_:.4f}")
print(f"Age + model-score log-likelihood:     {cph.log_likelihood_:.4f}")
print(f"Improvement in log-likelihood:        {cph.log_likelihood_ - cph_age.log_likelihood_:.4f}")

# -----------------------------
# 9) Optional: concordance comparison
# -----------------------------
print(f"\nAge-only Cox concordance index:       {cph_age.concordance_index_:.4f}")
print(f"Age + model-score Cox concordance:    {cph.concordance_index_:.4f}")
print(f"Delta concordance:                    {cph.concordance_index_ - cph_age.concordance_index_:.4f}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

# =========================================================
# Population-level risk curves and divergence
# with bootstrap 95% CI
# Using YOUR variable names
# =========================================================

# -----------------------------
# Settings
# -----------------------------
N_BOOT = 1000
RANDOM_STATE = 42
BIN_WIDTH = 2.0
MIN_N_PER_BIN = 5

# >>> REPLACED/ADDED: assign horizon once and reuse in Y-axis labels
RISK_HORIZON_YEARS = 5  #...***change this based on the KM curves.

# >>> REPLACED/ADDED: one shared figure size so both figures have identical dimensions
FIGSIZE = (9, 6)

# -----------------------------
# Helper functions
# -----------------------------
def predict_margin(model, X):
    return model.predict(xgb.DMatrix(X), output_margin=True)

def absolute_risk_from_margin(margin, S0_tstar):
    hr = np.exp(np.asarray(margin).ravel())
    return 1.0 - (S0_tstar ** hr)

def bootstrap_ci_mean(x, n_boot=200, alpha=0.05, rng=None):
    """
    Bootstrap CI for the mean of a 1D array.
    Returns: mean, lower, upper
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) == 0:
        return np.nan, np.nan, np.nan

    mean_x = np.mean(x)

    if len(x) == 1:
        return mean_x, mean_x, mean_x

    if rng is None:
        rng = np.random.default_rng(42)

    boot_means = np.empty(n_boot, dtype=float)
    n = len(x)

    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_means[b] = np.mean(x[idx])

    lower = np.percentile(boot_means, 100 * (alpha / 2))
    upper = np.percentile(boot_means, 100 * (1 - alpha / 2))
    return mean_x, lower, upper

# -----------------------------
# 1) Build plotting dataframe safely
# -----------------------------
pid_df = test_data_new[["PID"]].reset_index(drop=True).copy()
Xte_df = Xte.reset_index(drop=True).copy()

assert len(pid_df) == len(Xte_df), (
    f"Row mismatch: PID rows={len(pid_df)}, Xte rows={len(Xte_df)}"
)

plot_df = pd.concat([pid_df, Xte_df], axis=1)

# remove duplicate columns if any
plot_df = plot_df.loc[:, ~plot_df.columns.duplicated()].copy()

assert "AF_Age" in plot_df.columns, "AF_Age not found in plotting dataframe."

# full-model test risk from your pipeline
plot_df["risk_all"] = np.asarray(out["abs_risk_test"]).ravel()
plot_df["group"] = pd.Series(out["groups_test"]).reset_index(drop=True)

# -----------------------------
# 2) AF_Age-only predictions on all test rows
# -----------------------------
Xte_age = Xte_df.copy()
drop_cols = [c for c in feature_names if c != "AF_Age"]
Xte_age.loc[:, drop_cols] = bg_means.loc[drop_cols].values

lp_age = predict_margin(model_final, Xte_age)
risk_age = absolute_risk_from_margin(lp_age, out["S0_tstar"])

plot_df["risk_age_only"] = risk_age
plot_df["divergence"] = plot_df["risk_all"] - plot_df["risk_age_only"]

# -----------------------------
# 3) Bin AF_Age
# -----------------------------
age_min = np.floor(plot_df["AF_Age"].min())
age_max = np.ceil(plot_df["AF_Age"].max()) + BIN_WIDTH
bins = np.arange(age_min, age_max + BIN_WIDTH, BIN_WIDTH)

plot_df["AF_Age_bin"] = pd.cut(
    plot_df["AF_Age"],
    bins=bins,
    right=False,
    include_lowest=True
)

plot_df["AF_Age_mid"] = plot_df["AF_Age_bin"].apply(
    lambda x: x.left + (x.right - x.left) / 2 if pd.notna(x) else np.nan
).astype(float)

# -----------------------------
# 4) Bootstrap summary by AF_Age bin
# -----------------------------
rng = np.random.default_rng(RANDOM_STATE)

rows = []
for age_mid, g in plot_df.dropna(subset=["AF_Age_mid"]).groupby(
    "AF_Age_mid",
    observed=True
):
    n = len(g)
    if n < MIN_N_PER_BIN:
        continue

    mean_all, lo_all, hi_all = bootstrap_ci_mean(
        g["risk_all"].values,
        n_boot=N_BOOT,
        rng=rng
    )
    mean_age, lo_age, hi_age = bootstrap_ci_mean(
        g["risk_age_only"].values,
        n_boot=N_BOOT,
        rng=rng
    )
    mean_div, lo_div, hi_div = bootstrap_ci_mean(
        g["divergence"].values,
        n_boot=N_BOOT,
        rng=rng
    )

    rows.append({
        "AF_Age_mid": age_mid,
        "n": n,

        "mean_all": mean_all,
        "lo_all": lo_all,
        "hi_all": hi_all,

        "mean_age": mean_age,
        "lo_age": lo_age,
        "hi_age": hi_age,

        "mean_div": mean_div,
        "lo_div": lo_div,
        "hi_div": hi_div,
    })

summary = pd.DataFrame(rows).sort_values("AF_Age_mid").reset_index(drop=True)

print(summary.head())

# -----------------------------
# 5) Plot risk curves with bootstrap 95% CI
# -----------------------------
x = summary["AF_Age_mid"].values

mean_all = summary["mean_all"].values * 100
lo_all   = summary["lo_all"].values * 100
hi_all   = summary["hi_all"].values * 100

mean_age = summary["mean_age"].values * 100
lo_age   = summary["lo_age"].values * 100
hi_age   = summary["hi_age"].values * 100

# >>> REPLACED: plt.figure(figsize=(9, 6))
plt.figure(figsize=FIGSIZE)

plt.plot(x, mean_all, linewidth=2.5, label="All variables")
plt.fill_between(x, lo_all, hi_all, alpha=0.2)

plt.plot(x, mean_age, linewidth=2.5, linestyle="--", label="AF Age only")
plt.fill_between(x, lo_age, hi_age, alpha=0.2)

# Axis formatting
plt.xlim(50, x.max())
plt.xticks(np.arange(50, x.max() + 1, 10), fontsize=16)
plt.yticks(fontsize=16)

plt.xlabel("AF Age", fontsize=20)

# >>> REPLACED: plt.ylabel("Dementia risk (%)", fontsize=20)
plt.ylabel(f"{RISK_HORIZON_YEARS}-year Dementia risk (%)", fontsize=20)

plt.legend(fontsize=16)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("population_risk_full_vs_age_only.png", dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------
# 6) Plot divergence with bootstrap 95% CI
# -----------------------------
mean_div = summary["mean_div"].values * 100
lo_div   = summary["lo_div"].values * 100
hi_div   = summary["hi_div"].values * 100

# >>> REPLACED: plt.figure(figsize=(9, 5))
plt.figure(figsize=FIGSIZE)

plt.plot(x, mean_div, linewidth=2.5, label="Divergence (All - AF_Age)")
plt.fill_between(x, lo_div, hi_div, alpha=0.2)

plt.axhline(0, linestyle=":", linewidth=1)

# Axis formatting
plt.xlim(50, x.max())
plt.xticks(np.arange(50, x.max() + 1, 10), fontsize=16)
plt.yticks(fontsize=16)

plt.xlabel("AF Age", fontsize=20)

# >>> REPLACED: plt.ylabel("Difference in dementia risk (%)", fontsize=20)
plt.ylabel(f"Difference in {RISK_HORIZON_YEARS}-year Dementia risk (%)", fontsize=20)

plt.legend(fontsize=16)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("population_divergence_bootstrap_ci.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# =========================================================
# N-year dementia risk table at selected AF ages:
# MIND-AF model vs AF-age-only model
# =========================================================

TARGET_AF_AGES = [65, 75, 85]
RISK_HORIZON_YEARS = 5  # change this to N

risk_rows = []

for target_age in TARGET_AF_AGES:
    idx = (summary["AF_Age_mid"] - target_age).abs().idxmin()
    row = summary.loc[idx]

    p_all = row["mean_all"]
    p_age = row["mean_age"]

    risk_rows.append({
        "AF Age": target_age,
        "Closest AF Age Bin": row["AF_Age_mid"],

        f"{RISK_HORIZON_YEARS}-yr Risk MIND-AF (%)": p_all * 100,
        f"{RISK_HORIZON_YEARS}-yr Risk Age-only (%)": p_age * 100,

        # Absolute difference (what you already had)
        "Absolute Difference (%)": (p_all - p_age) * 100,

        # >>> ADD THIS: Relative Risk Ratio
        "Relative Risk (RR)": (p_all / p_age) if p_age > 0 else np.nan,

        # >>> OPTIONAL: Percent increase (more interpretable clinically)
        "Relative Increase (%)": ((p_all / p_age - 1) * 100) if p_age > 0 else np.nan,

        "N in Bin": int(row["n"])
    })

risk_table = pd.DataFrame(risk_rows).round(2)
risk_table

In [ ]:
# =========================================================
# WEIGHTED RRR with 95% CI (bootstrap over bins)
# =========================================================

N_BOOT = 1000
rng = np.random.default_rng(42)

weights = summary["n"].values
p_all_bins = summary["mean_all"].values
p_age_bins = summary["mean_age"].values

n_bins = len(p_all_bins)
rrr_boot_bins = np.empty(N_BOOT)

for b in range(N_BOOT):
    idx = rng.integers(0, n_bins, n_bins)  # resample bins

    w = weights[idx]
    mean_all_b = np.average(p_all_bins[idx], weights=w)
    mean_age_b = np.average(p_age_bins[idx], weights=w)

    rrr_boot_bins[b] = mean_all_b / mean_age_b

# Point estimate
RRR_w = np.average(p_all_bins, weights=weights) / np.average(p_age_bins, weights=weights)

# CI
lo_w = np.percentile(rrr_boot_bins, 2.5)
hi_w = np.percentile(rrr_boot_bins, 97.5)

# Relative %
RRR_w_pct = (RRR_w - 1) * 100
lo_w_pct = (lo_w - 1) * 100
hi_w_pct = (hi_w - 1) * 100

print("\n===== WEIGHTED RRR (Bin bootstrap) =====")
print(f"RRR: {RRR_w:.3f} (95% CI: {lo_w:.3f} – {hi_w:.3f})")
print(f"Relative increase: {RRR_w_pct:.1f}% (95% CI: {lo_w_pct:.1f}% – {hi_w_pct:.1f}%)")

### Relative risk Ratio and risks of MIND-AF versus age across age groups.

In [ ]:
# =========================================================
# Bar plot by clinical AF-age groups
# WITH:
#   - Absolute risk 95% CI whiskers
#   - RR + 95% CI on secondary axis
#   - Per-bin p-values
#   - Significance brackets + stars
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Settings
# -----------------------------
AGE_GROUP_ORDER = ["≤64", "65–74", "75–84", "≥85"]
AGE_GROUP_BINS = [-np.inf, 65, 75, 85, np.inf]

N_BOOT = 1000
N_PERM = 5000
RANDOM_STATE = 42
FIGSIZE = (10, 6)

rng = np.random.default_rng(RANDOM_STATE)

# -----------------------------
# Helper: mean CI
# -----------------------------
def bootstrap_ci_mean(x, n_boot=1000, alpha=0.05, rng=None):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    if rng is None:
        rng = np.random.default_rng(42)

    if len(x) == 0:
        return np.nan, np.nan, np.nan

    mean_x = np.mean(x)

    if len(x) == 1:
        return mean_x, mean_x, mean_x

    boot_means = np.empty(n_boot)

    for b in range(n_boot):
        idx = rng.integers(0, len(x), len(x))
        boot_means[b] = np.mean(x[idx])

    lo = np.percentile(boot_means, 2.5)
    hi = np.percentile(boot_means, 97.5)

    return mean_x, lo, hi


# -----------------------------
# Helper: RR CI
# -----------------------------
def bootstrap_ci_rr(risk_all, risk_age, n_boot=1000, alpha=0.05, rng=None):
    risk_all = np.asarray(risk_all, dtype=float)
    risk_age = np.asarray(risk_age, dtype=float)

    mask = np.isfinite(risk_all) & np.isfinite(risk_age)
    risk_all = risk_all[mask]
    risk_age = risk_age[mask]

    if rng is None:
        rng = np.random.default_rng(42)

    n = len(risk_all)

    if n == 0:
        return np.nan, np.nan, np.nan

    mean_all = np.mean(risk_all)
    mean_age = np.mean(risk_age)

    rr = mean_all / mean_age if mean_age > 0 else np.nan

    if n == 1 or mean_age <= 0:
        return rr, rr, rr

    boot_rr = np.empty(n_boot)

    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        m_all = np.mean(risk_all[idx])
        m_age = np.mean(risk_age[idx])
        boot_rr[b] = m_all / m_age if m_age > 0 else np.nan

    boot_rr = boot_rr[np.isfinite(boot_rr)]

    lo = np.percentile(boot_rr, 2.5)
    hi = np.percentile(boot_rr, 97.5)

    return rr, lo, hi


# -----------------------------
# Helper: paired permutation p-value
# -----------------------------
def paired_permutation_pvalue(x, y, n_perm=5000, rng=None):
    """
    Two-sided paired permutation/sign-flip test.
    Tests whether mean(x - y) differs from 0.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    d = x[mask] - y[mask]

    if rng is None:
        rng = np.random.default_rng(42)

    if len(d) < 2:
        return np.nan

    observed = np.abs(np.mean(d))

    null_stats = np.empty(n_perm)

    for i in range(n_perm):
        signs = rng.choice([-1, 1], size=len(d))
        null_stats[i] = np.abs(np.mean(d * signs))

    p = (np.sum(null_stats >= observed) + 1) / (n_perm + 1)

    return p


# -----------------------------
# Helper: p-value stars
# -----------------------------
def p_to_stars(p):
    if not np.isfinite(p):
        return "ns"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"


# -----------------------------
# Helper: draw significance bracket
# -----------------------------
def add_sig_bracket(ax, x1, x2, y, h, text, fontsize=16):
    ax.plot(
        [x1, x1, x2, x2],
        [y, y + h, y + h, y],
        color="black",
        linewidth=1.5
    )
    ax.text(
        (x1 + x2) / 2,
        y + h,
        text,
        ha="center",
        va="bottom",
        fontsize=fontsize,
        fontweight="bold"
    )


# -----------------------------
# 1) Create age groups
# -----------------------------
plot_df["AF_Age_group"] = pd.cut(
    plot_df["AF_Age"],
    bins=AGE_GROUP_BINS,
    labels=AGE_GROUP_ORDER,
    right=False,
    include_lowest=True
)

# -----------------------------
# 2) Compute stats per group
# -----------------------------
rows = []

for age_group, g in plot_df.dropna(subset=["AF_Age_group"]).groupby("AF_Age_group", observed=True):

    m_all, lo_all, hi_all = bootstrap_ci_mean(
        g["risk_all"],
        n_boot=N_BOOT,
        rng=rng
    )

    m_age, lo_age, hi_age = bootstrap_ci_mean(
        g["risk_age_only"],
        n_boot=N_BOOT,
        rng=rng
    )

    rr, rr_lo, rr_hi = bootstrap_ci_rr(
        g["risk_all"],
        g["risk_age_only"],
        n_boot=N_BOOT,
        rng=rng
    )

    pval = paired_permutation_pvalue(
        g["risk_all"],
        g["risk_age_only"],
        n_perm=N_PERM,
        rng=rng
    )

    rows.append({
        "group": str(age_group),
        "N": len(g),

        "mindaf_mean": m_all * 100,
        "mindaf_lo": lo_all * 100,
        "mindaf_hi": hi_all * 100,

        "age_mean": m_age * 100,
        "age_lo": lo_age * 100,
        "age_hi": hi_age * 100,

        "rr": rr,
        "rr_lo": rr_lo,
        "rr_hi": rr_hi,

        "p_value": pval,
        "stars": p_to_stars(pval)
    })

df = pd.DataFrame(rows)

df["group"] = pd.Categorical(
    df["group"],
    categories=AGE_GROUP_ORDER,
    ordered=True
)

df = df.sort_values("group").reset_index(drop=True)

display(df.round(4))


# -----------------------------
# 3) Plot
# -----------------------------
x = np.arange(len(df))
w = 0.35

fig, ax1 = plt.subplots(figsize=FIGSIZE)

# MIND-AF bars
ax1.bar(
    x - w / 2,
    df["mindaf_mean"],
    width=w,
    yerr=[
        df["mindaf_mean"] - df["mindaf_lo"],
        df["mindaf_hi"] - df["mindaf_mean"]
    ],
    capsize=6,
    label="MIND-AF",
    alpha=0.9
)

# AF-age-only bars
ax1.bar(
    x + w / 2,
    df["age_mean"],
    width=w,
    yerr=[
        df["age_mean"] - df["age_lo"],
        df["age_hi"] - df["age_mean"]
    ],
    capsize=6,
    label="AF age only",
    alpha=0.9
)

ax1.set_xlabel("AF Age Group", fontsize=20)
ax1.set_ylabel(f"{RISK_HORIZON_YEARS}-year Dementia Risk (%)", fontsize=20)
ax1.set_xticks(x)
ax1.set_xticklabels(df["group"].astype(str), fontsize=16)
ax1.tick_params(axis="y", labelsize=16)
ax1.grid(axis="y", alpha=0.3)

# -----------------------------
# 4) Add significance brackets
# -----------------------------
bar_tops = np.maximum(df["mindaf_hi"], df["age_hi"])
y_range = max(bar_tops.max(), 1) - 0

bracket_h = 0.03 * max(bar_tops.max(), 1)
bracket_offset = 0.06 * max(bar_tops.max(), 1)

for i in range(len(df)):
    y = bar_tops.iloc[i] + bracket_offset
    add_sig_bracket(
        ax=ax1,
        x1=x[i] - w / 2,
        x2=x[i] + w / 2,
        y=y,
        h=bracket_h,
        text=df.loc[i, "stars"],
        fontsize=16
    )

# expand y-axis so brackets fit
ax1.set_ylim(0, max(bar_tops + bracket_offset + bracket_h) * 1.18)


# -----------------------------
# 5) Secondary axis: RR
# -----------------------------
ax2 = ax1.twinx()

rr_err = np.vstack([
    df["rr"] - df["rr_lo"],
    df["rr_hi"] - df["rr"]
])

ax2.errorbar(
    x,
    df["rr"],
    yerr=rr_err,
    color="red",
    marker="o",
    markersize=10,
    linewidth=2.5,
    capsize=5,
    label="Relative Risk Ratio"
)

ax2.axhline(
    1.0,
    color="red",
    linestyle=":",
    linewidth=1.5
)

ax2.set_ylabel(
    "Relative Risk Ratio (MIND-AF / Age)",
    fontsize=20,
    color="red"
)

ax2.tick_params(axis="y", labelsize=16, colors="red")


# -----------------------------
# 6) Combined legend
# -----------------------------
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()

ax1.legend(
    h1 + h2,
    l1 + l2,
    fontsize=14,
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    f"age_group_barplot_with_ci_rr_pvalues_{RISK_HORIZON_YEARS}yr.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# If needed: !pip -q install shap
import numpy as np, pandas as pd, matplotlib.pyplot as plt, shap, xgboost as xgb
from typing import List

def xgb_feature_contributions_plot(
    booster, meta,
    X_bg: pd.DataFrame,               # background: e.g., TRAIN+VAL aligned to feature order
    X_test: pd.DataFrame,             # test features
    strata,                           # risk-group labels for averaging
    feature_cols: List[str] | None = None,   # if None, use all features
    top_k: int = 15,
    title: str = "Top drivers of risk (High − Low) — XGBoost Cox (SHAP)",
    save_path: str = "xgb_feature_contributions.png"  # new: save to file
):
    # 1) Align columns
    feats = meta["feature_names"]
    Xbg = X_bg.loc[:, feats].replace([np.inf, -np.inf], np.nan).dropna()
    Xte = X_test.loc[:, feats].copy()

    # Ensure strata is a categorical Series aligned with X_test
    Xte = Xte.reset_index(drop=True)
    if not isinstance(strata, pd.Series):
        strata = pd.Series(strata, index=Xte.index, name="risk_group")
    else:
        strata = strata.reset_index(drop=True)
    strata = strata.astype("category")

    # 2) Build SHAP explainer on raw margin (log-risk), with background
    expl = shap.TreeExplainer(
        booster,
        data=Xbg.values.astype(np.float32),
        feature_perturbation="interventional",
        model_output="raw"
    )

    # 3) Compute SHAP values on TEST (per-patient, per-feature contributions on log-risk scale)
    shap_vals = expl.shap_values(Xte.values.astype(np.float32))  # shape: (n_test, p)
    contrib = pd.DataFrame(shap_vals, columns=feats, index=Xte.index)

    # Optionally restrict to a feature subset
    if feature_cols is not None:
        keep = [c for c in feature_cols if c in contrib.columns]
        missing = sorted(set(feature_cols) - set(keep))
        if missing:
            print(f"[xgb_feature_contributions_plot] Skipping features not found: {missing}")
        if not keep:
            raise ValueError("None of the requested features are in the model.")
        contrib = contrib[keep]

    # 4) Average contributions within strata, then High − Low contrast
    cats = list(strata.cat.categories.astype(str))
    try:
        lbl_low  = next(l for l in cats if "Low"  in l)
        lbl_high = next(l for l in cats if "High" in l)
    except StopIteration:
        total = contrib.sum(axis=1)
        means = total.groupby(strata).mean().sort_values()
        lbl_low, lbl_high = str(means.index[0]), str(means.index[-1])

    mean_by_grp = contrib.groupby(strata).mean().T
    diff = (mean_by_grp[str(lbl_high)] - mean_by_grp[str(lbl_low)]).sort_values(ascending=False)

    top = diff.head(top_k)

    # ---- Pretty-label renaming for the plot only ----
    pretty_map = {
        "AF_age_4": "AF Age (>=85)",
        "AF_age_3": "AF Age (75-84)",
    }
    plot_idx = [pretty_map.get(i, i) for i in top.index]

    # 5) Plot
    plt.figure(figsize=(9,6))
    plt.barh(plot_idx, top.values)
    plt.gca().invert_yaxis()  # highest at the top

    plt.xlabel("Mean contribution on log-risk (SHAP): High − Low", fontsize=14)
    plt.ylabel("Feature", fontsize=14)
    plt.title(title, fontsize=14)
    plt.tick_params(axis='both', which='major', labelsize=12)
    plt.tight_layout()

    # ---- Save to file ----
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    return mean_by_grp, diff


In [ ]:
# Background = TRAIN+VAL (recommended)
X_bg = pd.concat([X_train, X_val], axis=0)

# If you want to mirror the CoxPH feature list:
# feature_cols = selected_cols  # or pass None to use all features
risk_groups_test = out["groups_test"]
mean_by_grp_shap, diff_shap = xgb_feature_contributions_plot(
    booster=model_final, meta=meta,
    X_bg=X_bg,
    X_test=X_test, strata=risk_groups_test,   # your Low/Med/High labels on TEST
    feature_cols=None,                        # or a list if you want to restrict
    top_k=15,
    title="Top drivers of risk (High − Low) — XGBoost Cox (SHAP)",
    save_path= "xgb_feature_contributions_model3_for1.png"  #...CHANGE the model number.
)


In [ ]:
cols = mean_by_grp_shap.columns.tolist()

group_col_map = {
    "Low":    [c for c in cols if c.startswith("Low")][0],
    "Medium": [c for c in cols if c.startswith("Medium")][0],
    "High":   [c for c in cols if c.startswith("High")][0],
}

# Clean group labels (used in legend)
clean_group_order = ["Low", "Medium", "High"]

# Actual columns used for data extraction
actual_group_order = [group_col_map[g] for g in clean_group_order]

In [ ]:
mean_by_grp_shap.index

In [ ]:
# Identify group labels (edit these if your labels differ)
#low    = "Low (≤2%)"
#med    = "Medium (2–7.3%)"
#high   = "High (≥7.3%)"

strata = out["groups_test"]
cats = list(strata.cat.categories.astype(str))
low  = next(l for l in cats if "Low"  in l)
med = next(l for l in cats if "Medium" in l)
high = next(l for l in cats if "High" in l)
print(low, med, high)

# 1) Low → Medium
diff_low_med = (mean_by_grp_shap[med] - mean_by_grp_shap[low]).sort_values(ascending=False)

# 2) Medium → High
diff_med_high = (mean_by_grp_shap[high] - mean_by_grp_shap[med]).sort_values(ascending=False)

# 3) Low → High
diff_low_high = (mean_by_grp_shap[high] - mean_by_grp_shap[low]).sort_values(ascending=False)

# Print top drivers for each transition
print("\n=== Drivers of Risk: Low → Medium ===")
print(diff_low_med.head(15))

print("\n=== Drivers of Risk: Medium → High ===")
print(diff_med_high.head(15))

print("\n=== Drivers of Risk: Low → High ===")
print(diff_low_high.head(15))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_two_panel_contrib_spacing(
    mean_by_grp_shap: pd.DataFrame,
    group_low: str = "Low",
    group_med: str = "Medium",
    group_high: str = "High",
    top_k: int = 20,
    category_map: dict[str, str] | None = None,
    pretty_map: dict[str, str] | None = None,
    title_left: str = "Drivers of Risk: Low → Medium",
    title_right: str = "Drivers of Risk: Medium → High",
    xlabel: str = "Mean contribution to XGBoost log-risk (SHAP)",
    save_path: str | None = None,
    figsize=(12, 8),
    hard_cap: float | None = 0.4,
    # NEW: layout controls for spacing
    bottom_pad: float = 0.16,      # reserve % of figure height at bottom for legend
    legend_y: float = 0.06,        # legend vertical position in figure coords (0=bottom)
    legend_ncol: int = 6,
    font_size: int = 14,
):
    # --- checks ---
    for col in [group_low, group_med, group_high]:
        if col not in mean_by_grp_shap.columns:
            raise ValueError(f"Missing column '{col}' in mean_by_grp_shap.")

    # --- diffs ---
    diff_lm = mean_by_grp_shap[group_med] - mean_by_grp_shap[group_low]
    diff_mh = mean_by_grp_shap[group_high] - mean_by_grp_shap[group_med]

    top_lm = diff_lm.abs().sort_values(ascending=False).head(top_k).index
    top_mh = diff_mh.abs().sort_values(ascending=False).head(top_k).index

    # DESCENDING so biggest bars are first; we keep invert_yaxis() so they appear on top
    s_lm = diff_lm.loc[top_lm].sort_values(ascending=False)
    s_mh = diff_mh.loc[top_mh].sort_values(ascending=False)

    # --- helpers ---
    def _norm(s: str | None):
        if s is None: return None
        return "".join(ch for ch in s.lower() if ch.isalnum())

    palette = {
        "age": "black",
        "cardiovascular": "red",
        "metabolic": "orange",
        "neurocognitive": "green",
        "sociodemographic": "blue",
        "other": "gray",
    }
    pretty_cat = {
        "age": "Age",
        "cardiovascular": "Cardiovascular",
        "metabolic": "Metabolic",
        "neurocognitive": "Neurocognitive",
        "sociodemographic": "Socio-demographic",
        "other": "Other",
    }

    def feat_color(feat: str):
        if not category_map: return "gray"
        cat = _norm(category_map.get(feat, "other"))
        return palette.get(cat, "gray")

    def feat_label(feat: str):
        if not pretty_map: return feat
        return pretty_map.get(feat, feat)

    y_lm = [feat_label(f) for f in s_lm.index]
    y_mh = [feat_label(f) for f in s_mh.index]
    c_lm = [feat_color(f) for f in s_lm.index]
    c_mh = [feat_color(f) for f in s_mh.index]

    # --- figure ---
    fig, axes = plt.subplots(1, 2, figsize=figsize, sharex=False)

    def draw_panel(ax, series, ylabels, colors, title):
        ax.barh(ylabels, series.values, color=colors, edgecolor="none")
        ax.axvline(0, color="#999", lw=1)
        ax.set_title(title, fontsize=font_size, pad=10)
        ax.set_xlabel(xlabel, fontsize=font_size)
        ax.tick_params(axis='both', labelsize=font_size)
        ax.invert_yaxis()

        if hard_cap is not None:
            xmax = float(hard_cap); xmin = -xmax
            ax.set_xlim(xmin, xmax)
            for yi, v in enumerate(series.values):
                if v > xmax:
                    ax.text(xmax, yi, f" ▶ {v:.3f}", va="center", ha="left", fontsize=font_size-5)
                elif v < xmin:
                    ax.text(xmin, yi, f"◀ {v:.3f} ", va="center", ha="right", fontsize=font_size-5)

    draw_panel(axes[0], s_lm, y_lm, c_lm, title_left)
    draw_panel(axes[1], s_mh, y_mh, c_mh, title_right)

    # --- legend below subplots with guaranteed space ---
    if category_map:
        cats_used = {_norm(category_map.get(f, "other")) for f in set(s_lm.index) | set(s_mh.index)}
        handles, labels = [], []
        for c in sorted(cats_used):
            handles.append(plt.Line2D([0],[0], marker='s', color='w',
                                      markerfacecolor=palette.get(c, "gray"),
                                      markersize=10))
            labels.append(pretty_cat.get(c, "Other"))

        # Reserve bottom margin for legend, then place it in figure coordinates
        fig.tight_layout(rect=[0, bottom_pad, 1, 1])  # leave bottom_pad space
        fig.legend(handles, labels,
                   loc="lower center",
                   ncol=min(legend_ncol, len(labels)),
                   frameon=False,
                   bbox_to_anchor=(0.5, legend_y),
                   bbox_transform=fig.transFigure,
                   fontsize=font_size)
    else:
        fig.tight_layout(rect=[0, bottom_pad, 1, 1])

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    return {
        "diff_low_med_top": s_lm.sort_values(ascending=False),
        "diff_med_high_top": s_mh.sort_values(ascending=False),
    }


In [ ]:
# Example category mapping (optional). Map each feature to a category label.
category_map = {
    "AF_age_3": "age",
    "AF_age_4": "age",
    "AF_age_2": "age",
    "AF_Age": "age",
    "weight": "metabolic",
    "bmi": "metabolic",
    "HTN": "cardiovascular",
    "DM": "metabolic",
    "CKD": "metabolic",
    "OSA": "metabolic",
    "TIA_CVA_Stroke": "cardiovascular",
    "Cognitive_Impairment": "neurocognitive",
    "Cognitive_Deficit": "neurocognitive",
    "Parkinson": "neurocognitive",
    "Osteoporosis": "other",
    "Osteoarthritis": "other",
    "QRS Value 3mo": "cardiovascular",
    "RR Value 3mo": "cardiovascular",
    "Depression": "neurocognitive",
    "Sodium Value 3mo": "metabolic",
    "Calcium Value 3mo": "metabolic",
    "Creatinine Value 3mo": "metabolic",
    "Calcium_missing": "other",
    "HCT_missing": "other",
    "Marital": "sociodemographic",
    "Insurance": "sociodemographic",
    "height": "other",
}

# Optional pretty names (for axis only)
pretty_map = {
        "AF_age_3": "AF Age (75–84)",
        "AF_age_4": "AF Age (≥85)",
        "AF_age_2": "AF Age (65–74)",
        "bmi": "BMI",
        "DM": "Diabetes",
        "HTN": "Hypertension",
        "CKD": "Chronic Kidney Disease",
        "TIA_CVA_Stroke": "Prior TIA/CVA",
        "OSA": "Obstructive Sleep Apnea",
        "QRS Value 3mo": "QRS Axis (3 mo)",
        "Sodium Value 3mo": "Sodium (3 mo)",
        "Calcium Value 3mo": "Calcium (3 mo)",
        "PPI": "Proton Pump Inhibitors",
        "Calcium_missing": "Calcium Missing",
        "HCT_missing": "Hematocrit Missing",
        "Cognitive_Impairment": "Cognitive Impairment",
        "Marital": "Divorced/Widowed"
    }

for col in mean_by_grp_shap.columns:
  if "Low" in col:
    low = col
  elif "Medium" in col:
    med = col
  elif "High" in col:
    high = col

result = plot_two_panel_contrib_spacing(
    mean_by_grp_shap,
    group_low=low,
    group_med=med,
    group_high=high,
    top_k=10,
    category_map=category_map,   # or None
    pretty_map=pretty_map,       # or None
    hard_cap=0.2,                # ← cap at ±0.4
    save_path="two_panel_contrib_capped_model3_for1.png", #...put the model number here.
    figsize=(13, 8),
    bottom_pad = 0.12
)

# Access the exact numbers used in the panels (for tables or supplement)
result["diff_low_med_top"].head(15)
result["diff_med_high_top"].head(15)

#Use these components: "Age": Black, "Cardiovascular": Red, "Neurocognitive": Green, "Socio-demographic":Blue, "Other":Gray.


#Landmark Analysis

In [ ]:
# =========================================================
# Landmark analysis (t*) using the FROZEN Cox model's absolute risk
# (no new classifier is trained; we threshold 1 - S0(t)^exp(margin))
# =========================================================
import numpy as np, pandas as pd, os
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

def _margin(booster, X, feats):
    return booster.predict(xgb.DMatrix(X.loc[:, feats]), output_margin=True)

def _breslow_S0(y_df, risk):
    """Breslow baseline survival S0(t) from (time,event) and risk=exp(margin)."""
    d = y_df.reset_index(drop=True)
    order = np.argsort(d["time"].values)
    T, E = d["time"].values[order], d["event"].astype(int).values[order]
    r = np.asarray(risk).ravel()[order]
    uniq_t, idx = np.unique(T, return_index=True)
    rev = np.cumsum(r[::-1])[::-1]
    dH = np.where(rev[idx] > 0, np.add.reduceat(E, idx) / rev[idx], 0.0)
    return uniq_t, np.exp(-np.cumsum(dH))

def _S0_at(uniq_t, S0, t_star):
    if t_star <= uniq_t[0]:  return 1.0
    if t_star >= uniq_t[-1]: return float(S0[-1])
    return float(S0[np.searchsorted(uniq_t, t_star, side="right") - 1])

def binary_event_at_t(y_df, t_star, drop_early_censored=True):
    """Landmark label: dementia by t*; drop patients censored before t*."""
    y = y_df.reset_index(drop=True)
    lab = ((y["event"] == 1) & (y["time"] <= t_star)).astype(int).to_numpy()
    keep = (~((y["event"] == 0) & (y["time"] < t_star))).to_numpy() if drop_early_censored \
           else np.ones(len(y), bool)
    return lab[keep], keep

def _metrics_at(y, proba, thr):
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens = tp/(tp+fn) if (tp+fn) else np.nan
    spec = tn/(tn+fp) if (tn+fp) else np.nan
    ppv  = tp/(tp+fp) if (tp+fp) else np.nan
    f1   = 2*ppv*sens/(ppv+sens) if (ppv and sens and (ppv+sens) > 0) else np.nan
    return sens, spec, ppv, f1

def _boot_ci(fn, y, proba, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed); n = len(y); v = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2: continue
        v.append(fn(y[idx], proba[idx]))
    return np.nanpercentile(v, [2.5, 97.5])

def landmark_frozen_cox(
    booster, meta,
    X_fit, y_fit,        # for the Breslow baseline (train+val)
    X_eval, y_eval,      # cohort to evaluate the landmark on (test)
    t_star=5.0,
    thresholds=(0.045, 0.02),
    n_boot=1000, seed=42, model=1, out_dir="/content",
):
    os.makedirs(out_dir, exist_ok=True)
    feats = meta["feature_names"]

    # 1. Breslow baseline on the FIT set, S0 at t*
    uniq_t, S0 = _breslow_S0(y_fit, np.exp(_margin(booster, X_fit, feats)))
    S0_t = _S0_at(uniq_t, S0, t_star)

    # 2. Frozen-model 5y absolute risk on the EVAL set
    risk = 1.0 - (S0_t ** np.exp(_margin(booster, X_eval, feats)))

    # 3. Landmark eligibility (drop early-censored)
    y, keep = binary_event_at_t(y_eval, t_star)
    proba = risk[keep]
    print(f"--- Landmark {t_star:g}-Year (frozen Cox model) ---")
    print(f"S0({t_star:g}y)={S0_t:.4f} | eligible {len(y):,} "
          f"(dropped {int((~keep).sum()):,} early-censored) | "
          f"events {int(y.sum()):,} ({100*y.mean():.2f}%)")

    # 4. AUROC (threshold-free) + CI
    auc = roc_auc_score(y, proba)
    auc_ci = _boot_ci(lambda a, b: roc_auc_score(a, b), y, proba, n_boot, seed)
    print(f"AUROC: {auc:.4f} ({auc_ci[0]:.4f}-{auc_ci[1]:.4f})")

    # 5. Operating points at FIXED thresholds (NOT tuned on the eval set)
    rows = []
    for thr in thresholds:
        sens, spec, ppv, f1 = _metrics_at(y, proba, thr)
        s  = _boot_ci(lambda a, b: _metrics_at(a, b, thr)[0], y, proba, n_boot, seed)
        sp = _boot_ci(lambda a, b: _metrics_at(a, b, thr)[1], y, proba, n_boot, seed)
        p  = _boot_ci(lambda a, b: _metrics_at(a, b, thr)[2], y, proba, n_boot, seed)
        f  = _boot_ci(lambda a, b: _metrics_at(a, b, thr)[3], y, proba, n_boot, seed)
        print(f"  risk>={100*thr:g}%: Sens {sens:.3f} ({s[0]:.3f}-{s[1]:.3f}) | "
              f"Spec {spec:.3f} ({sp[0]:.3f}-{sp[1]:.3f}) | "
              f"PPV {ppv:.3f} ({p[0]:.3f}-{p[1]:.3f}) | F1 {f1:.3f} ({f[0]:.3f}-{f[1]:.3f})")
        r3 = lambda x: round(float(x), 3)
        rows.append({"threshold_risk": thr, "flagged": int((proba >= thr).sum()),
                     "AUROC": r3(auc), "AUROC_lo": r3(auc_ci[0]), "AUROC_hi": r3(auc_ci[1]),
                     "sensitivity": r3(sens), "sens_lo": r3(s[0]), "sens_hi": r3(s[1]),
                     "specificity": r3(spec), "spec_lo": r3(sp[0]), "spec_hi": r3(sp[1]),
                     "ppv": r3(ppv), "ppv_lo": r3(p[0]), "ppv_hi": r3(p[1]),
                     "f1": r3(f1), "f1_lo": r3(f[0]), "f1_hi": r3(f[1])})
    perf = pd.DataFrame(rows)
    perf.to_csv(os.path.join(out_dir, f"landmark_frozen_model{model}_t{t_star}.csv"), index=False)

    # 6. ROC with shaded 95% CI band
    fpr, tpr, _ = roc_curve(y, proba)
    mean_fpr = np.linspace(0, 1, 100)
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    tprs = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(pos, len(pos), True), rng.choice(neg, len(neg), True)])
        fb, tb, _ = roc_curve(y[idx], proba[idx])
        it = np.interp(mean_fpr, fb, tb); it[0] = 0.0; tprs.append(it)
    tprs = np.array(tprs)
    lo, hi = np.percentile(tprs, 2.5, axis=0), np.percentile(tprs, 97.5, axis=0)

    plt.figure(figsize=(7, 6.5))
    plt.fill_between(mean_fpr, lo, hi, color="royalblue", alpha=0.2, label="95% CI")
    plt.plot(fpr, tpr, color="royalblue", lw=2, label=f"XGB {t_star:g}yr (AUROC={auc:.3f})")
    plt.plot([0, 1], [0, 1], color="gray", ls="--", lw=1)
    plt.xlim([-0.01, 1.0]); plt.ylim([0.0, 1.01])
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"Landmark ROC Curve ({t_star:g} Years)")
    plt.legend(loc="lower right"); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"landmark_roc_frozen_model{model}_t{t_star}.png"), dpi=300)
    plt.show()
    return {"S0_tstar": S0_t, "auc": auc, "auc_ci": auc_ci, "perf": perf,
            "y": y, "proba": proba}

In [ ]:
# -------- execution (frozen Cox model; BIC model = booster, meta) --------
res_lm = landmark_frozen_cox(
    booster=model_final, meta=meta,          # <- use "booster" for BIC model; use "model_final"/"meta" for the 80-var model
    X_fit=pd.concat([X_train, X_val], axis=0),
    y_fit=pd.concat([y_train, y_val], axis=0),
    X_eval=X_test, y_eval=y_test,
    t_star=5.0,
    thresholds=(0.048, 0.02),  #...0.045 for booster, and 0.052 for model_final. 0.048 for the ancillary model_final(s)
    model=1,
)
res_lm["perf"]